# A Machine Learning Approach to Detect Fraud and Cyber Attacks

**Client:** XYZ Cybersecurity (early-stage startup, network threat detection)
**Task:** build and validate a supervised model that flags fraudulent / malicious
network flows accurately enough to be used inside a Security Operations Centre (SOC).

---

## Business framing

XYZ Cybersecurity monitors network traffic on behalf of its customers. Every
*flow* (a bidirectional conversation between two endpoints) is summarised into
~78 statistical features by a flow meter (CICFlowMeter). The analyst team cannot
triage millions of flows per day by hand, so the goal is a model that:

| Requirement | Why it matters | How it is measured here |
| --- | --- | --- |
| Catch attacks | A missed intrusion (false negative) can mean data loss or fraud | Recall on the attack class, cost-weighted threshold |
| Do not drown analysts | Each false positive costs analyst time and may block a paying customer | Precision, false-positive rate, alert volume |
| Rank, not just label | SOC queues are ordered by risk | PR-AUC / ROC-AUC on predicted probability |
| Run in near-real-time | Inline detection needs sub-millisecond scoring | Measured inference latency & throughput |
| Be explainable & auditable | Analysts must justify a block; regulators require it | Feature importances, per-alert explanations, model card |

**Prediction target.** The raw data carries a multi-class `Label`
(`BENIGN` plus specific attack names). The primary model is **binary**:
`is_attack = 0` for `BENIGN`, `1` for anything else. This matches the
operational question ("should a human look at this flow?"), keeps the rare
attack families usable instead of splitting them into unlearnably small classes,
and is complemented later by a per-attack-family breakdown and a multi-class
extension (Section 6).

---

## Notebook roadmap

| Section | Marking criterion |
| --- | --- |
| 3. Data understanding, quality assessment and preparation | Data Understanding & Preparation |
| 4. Feature engineering | Data Understanding & Preparation |
| 5. Model selection (8 candidates, cross-validated) | Model Selection |
| 6. Performance measurement (metrics, curves, cost-based threshold) | Performance Measurement |
| 7. Hyperparameter tuning (randomised search, two models) | Hyperparameter Tuning |
| 8. Extra features and engineering considerations | Extra feature & consideration |
| 9. AI ethics considerations (+ quantitative fairness audit) | AI Ethics consideration |
| 10. Conclusions, limitations, future work | Documentation & Code Quality |

---

## How to run

1. `Runtime → Change runtime type → CPU` (a GPU is not required; the models are
   CPU based). A high-RAM runtime is helpful but not required.
2. Place `Dataset1.csv`, `Dataset2.csv`, `Dataset3.csv` in the root of *My Drive*.
3. `Runtime → Run all`. Expected end-to-end runtime on a standard 2-core Colab
   CPU runtime is roughly **25-40 minutes**; the cross-validated model comparison
   and the two randomised searches dominate. Every expensive cell is marked, and
   `Config` (next cell) exposes the sample sizes and search budget if you want to
   trade runtime for precision in either direction.
4. Every result the notebook discusses is printed or plotted by the cell above
   the discussion, and the final table in Section 10 aggregates all experiments.

> **Provenance note.** All numbers, tables and figures in this notebook are
> produced by the cells themselves - nothing is hard-coded. If a cell shows no
> output, it has not been executed yet in your runtime: use *Run all* to
> regenerate the complete set of outputs on your copy of the data.

**Reproducibility.** A single `RANDOM_STATE` seeds every split, sampler and
estimator; `Config` (next cell) holds all tunable constants so the whole
notebook can be re-run at a different sample size or cost ratio by editing one
place.

---
# Section 0 - Environment and global configuration

In [ ]:
# =============================================================================
# Libraries, display settings and global configuration
# =============================================================================
from __future__ import annotations

import os
import platform
import sys
import time
import warnings
from dataclasses import asdict, dataclass

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from IPython.display import display

warnings.filterwarnings("ignore")

# --- display -----------------------------------------------------------------
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update(
    {"figure.dpi": 110, "figure.figsize": (9.0, 4.5), "axes.titleweight": "bold"}
)


# --- configuration -----------------------------------------------------------
@dataclass(frozen=True)
class Config:
    '''Every tunable constant used in this notebook, in one auditable place.'''

    random_state: int = 42
    test_size: float = 0.25          # held-out fraction for final evaluation
    model_sample: int = 250_000      # stratified rows used for train+test
    cv_sample: int = 60_000          # rows used for model selection / tuning
    knn_sample: int = 25_000         # k-NN is O(n) per prediction
    cv_folds: int = 5                # folds for model selection
    tuning_folds: int = 3            # folds inside the randomised searches
    tuning_iters: int = 12           # sampled configurations per search
    corr_drop_threshold: float = 0.98
    cost_false_negative: float = 100.0   # missed attack
    cost_false_positive: float = 1.0     # wasted analyst triage
    artefact_dir: str = "artefacts"


CFG = Config()
RANDOM_STATE = CFG.random_state
np.random.seed(RANDOM_STATE)
os.makedirs(CFG.artefact_dir, exist_ok=True)

# Central registry: every experiment appends its metrics here so that Section 10
# can print one consolidated comparison table.
RESULTS: dict[str, dict] = {}

print("Python           :", sys.version.split()[0], "|", platform.platform())
print("numpy            :", np.__version__)
print("pandas           :", pd.__version__)
print("scikit-learn     :", sklearn.__version__)
print("joblib           :", joblib.__version__)
print("seaborn          :", sns.__version__)
print("\nConfiguration:")
for key, value in asdict(CFG).items():
    print(f"  {key:<22} = {value}")

---
# Section 1 - Loading the raw data

The three CSV exports are read from Google Drive. They are separate capture
sessions from the same monitored network, so they share one schema and can be
concatenated into a single modelling table.

In [8]:
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Load the datasets
df1 = pd.read_csv('/content/drive/MyDrive/Dataset1.csv')
df2 = pd.read_csv('/content/drive/MyDrive/Dataset2.csv')
df3 = pd.read_csv('/content/drive/MyDrive/Dataset3.csv')

# Verify they loaded correctly
print("Dataset 1 shape:", df1.shape)
print("Dataset 2 shape:", df2.shape)
print("Dataset 3 shape:", df3.shape)

# Display the first few rows of each dataset
display(df1.head())
display(df2.head())
display(df3.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset 1 shape: (692703, 79)
Dataset 2 shape: (170366, 79)
Dataset 3 shape: (191033, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,38308,1,1,6,6,6,6,6.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,389,479,11,5,172,326,79,0,15.636364,31.449238,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,88,1095,10,6,3150,3150,1575,0,315.000000,632.561635,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,389,15206,17,12,3452,6660,1313,0,203.058823,425.778474,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,88,1092,9,6,3150,3152,1575,0,350.000000,694.509719,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,389,113095465,48,24,9668,10012,403,0,201.416667,203.548293,...,32,203985.500,5.758373e+05,1629110,379,13800000.0,4.277541e+06,16500000,6737603,BENIGN
1,389,113473706,68,40,11364,12718,403,0,167.117647,171.919413,...,32,178326.875,5.034269e+05,1424245,325,13800000.0,4.229413e+06,16500000,6945512,BENIGN
2,0,119945515,150,0,0,0,0,0,0.000000,0.000000,...,0,6909777.333,1.170000e+07,20400000,6,24400000.0,2.430000e+07,60100000,5702188,BENIGN
3,443,60261928,9,7,2330,4221,1093,0,258.888889,409.702161,...,20,0.000,0.000000e+00,0,0,0.0,0.000000e+00,0,0,BENIGN
4,53,269,2,2,102,322,51,51,51.000000,0.000000,...,32,0.000,0.000000e+00,0,0,0.0,0.000000e+00,0,0,BENIGN


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,3268,112740690,32,16,6448,1152,403,0,201.5,204.724205,...,32,3.594286e+02,1.199802e+01,380,343,16100000.0,4.988048e+05,16400000,15400000,BENIGN
1,389,112740560,32,16,6448,5056,403,0,201.5,204.724205,...,32,3.202857e+02,1.574499e+01,330,285,16100000.0,4.987937e+05,16400000,15400000,BENIGN
2,0,113757377,545,0,0,0,0,0,0.0,0.000000,...,0,9.361829e+06,7.324646e+06,18900000,19,12200000.0,6.935824e+06,20800000,5504997,BENIGN
3,5355,100126,22,0,616,0,28,28,28.0,0.000000,...,32,0.000000e+00,0.000000e+00,0,0,0.0,0.000000e+00,0,0,BENIGN
4,0,54760,4,0,0,0,0,0,0.0,0.000000,...,0,0.000000e+00,0.000000e+00,0,0,0.0,0.000000e+00,0,0,BENIGN


---
# Section 2 - Schema harmonisation

Three defects are common in CICFlowMeter exports and all three break naive
concatenation, so they are fixed before anything else:

1. **Leading/trailing spaces in column names** (`' Flow Duration'`). Left alone
   these silently produce `KeyError`s and duplicate columns after `concat`.
2. **Repeated column names** (`Fwd Header Length` appears twice in some
   exports). `pandas` keeps both; the second is dropped here.
3. **Column order / membership differences** between capture days. The
   intersection is taken and re-ordered consistently, and any non-shared column
   is reported rather than silently discarded.

A `source_file` column is added, which is later used for the cross-capture
generalisation test (Section 8) and the fairness audit (Section 9).

In [ ]:
# =============================================================================
# Harmonise the three exports and build one modelling table
# =============================================================================
RAW_FRAMES = {"Dataset1": df1, "Dataset2": df2, "Dataset3": df3}


def harmonise(frame: pd.DataFrame, source: str) -> pd.DataFrame:
    '''Strip column names, drop repeated column names, tag the source file.'''
    out = frame.copy()
    out.columns = [str(c).strip() for c in out.columns]
    duplicated = out.columns[out.columns.duplicated()].unique().tolist()
    if duplicated:
        print(f"  {source}: dropping repeated column name(s) {duplicated}")
    out = out.loc[:, ~out.columns.duplicated(keep="first")]
    out["source_file"] = source
    return out


print("Harmonising schemas")
frames = {name: harmonise(frame, name) for name, frame in RAW_FRAMES.items()}

column_sets = [set(f.columns) for f in frames.values()]
common = set.intersection(*column_sets)
union = set.union(*column_sets)
print(f"\nColumns per file : {[f.shape[1] for f in frames.values()]}")
print(f"Shared columns   : {len(common)}")
print(f"Non-shared       : {sorted(union - common) if union - common else 'none'}")

# Canonical ordering taken from the first file, restricted to shared columns.
ordered = [c for c in frames["Dataset1"].columns if c in common]
df = pd.concat([f[ordered] for f in frames.values()], ignore_index=True)
del frames

print(f"\nCombined table   : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Memory footprint : {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")
print("\nColumn dtypes:")
print(df.dtypes.value_counts().to_string())
print("\nRows contributed by each source file:")
print(df["source_file"].value_counts().to_string())

---
# Section 3 - Data understanding and preparation

## 3.1 Target definition and class balance

`Label` is normalised first: some exports encode `Web Attack – Brute Force` with
a Unicode en-dash and inconsistent internal spacing, which would otherwise
create several "different" labels for one attack.

Two derived targets are created:

* `is_attack` - the **binary modelling target** (0 = benign, 1 = attack).
* `attack_family` - a coarse grouping (DoS, DDoS, Web Attack, Bot, ...) used for
  stratification, for per-family recall reporting and for the multi-class
  extension. Rare, closely-related variants are grouped so that each family is
  large enough to reason about.

In [ ]:
# =============================================================================
# Target definition: normalise labels, derive binary target and attack family
# =============================================================================
LABEL_COL = "Label"
BENIGN = "BENIGN"

df[LABEL_COL] = (
    df[LABEL_COL]
    .astype(str)
    .str.replace("\u2013", "-", regex=False)   # en-dash -> hyphen
    .str.replace("\u2014", "-", regex=False)   # em-dash -> hyphen
    .str.replace(r"\s+", " ", regex=True)      # collapse whitespace
    .str.strip()
    .str.upper()
)

FAMILY_RULES = [
    ("DDOS", "DDoS"),
    ("DOS", "DoS"),
    ("PORTSCAN", "Port Scan"),
    ("WEB ATTACK", "Web Attack"),
    ("PATATOR", "Brute Force"),
    ("BRUTE FORCE", "Brute Force"),
    ("BOT", "Botnet"),
    ("INFILTRATION", "Infiltration"),
    ("HEARTBLEED", "Heartbleed"),
]


def to_family(label: str) -> str:
    '''Map a fine-grained label onto a coarse attack family.'''
    if label == BENIGN:
        return "Benign"
    for pattern, family in FAMILY_RULES:
        if pattern in label:
            return family
    return "Other attack"


df["is_attack"] = (df[LABEL_COL] != BENIGN).astype("int8")
df["attack_family"] = df[LABEL_COL].map(to_family).astype("category")

# --- composition tables ------------------------------------------------------
composition = (
    pd.crosstab(df[LABEL_COL], df["source_file"])
    .assign(Total=lambda t: t.sum(axis=1))
    .sort_values("Total", ascending=False)
)
composition["% of data"] = 100 * composition["Total"] / len(df)
print("Label composition by source file")
display(composition)

print("\nAttack family composition")
family = df["attack_family"].value_counts().to_frame("rows")
family["% of data"] = 100 * family["rows"] / len(df)
display(family)

n_attack = int(df["is_attack"].sum())
n_benign = len(df) - n_attack
print(f"\nBinary target: benign = {n_benign:,} ({100 * n_benign / len(df):.2f}%)")
print(f"               attack = {n_attack:,} ({100 * n_attack / len(df):.2f}%)")
print(f"Imbalance ratio (benign : attack) = {n_benign / max(n_attack, 1):.1f} : 1")
print(f"Distinct labels = {df[LABEL_COL].nunique()}, "
      f"rarest = {composition['Total'].min():,} rows")

In [ ]:
# =============================================================================
# Visualising the class structure
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

# (a) fine-grained label counts, log scale: shows the several orders of
#     magnitude between BENIGN and the rarest attacks.
counts = df[LABEL_COL].value_counts().sort_values()
colours = ["#2a9d8f" if lab == BENIGN else "#e63946" for lab in counts.index]
axes[0].barh(counts.index, counts.values, color=colours)
axes[0].set_xscale("log")
axes[0].set_title("(a) Rows per label (log scale)")
axes[0].set_xlabel("rows")
for y, v in enumerate(counts.values):
    axes[0].text(v * 1.15, y, f"{v:,}", va="center", fontsize=7)

# (b) binary balance
binary = df["is_attack"].value_counts().sort_index()
axes[1].bar(["Benign (0)", "Attack (1)"], binary.values, color=["#2a9d8f", "#e63946"])
axes[1].set_title("(b) Binary target balance")
axes[1].set_ylabel("rows")
for x, v in enumerate(binary.values):
    axes[1].text(x, v, f"{v:,}\n{100 * v / len(df):.1f}%", ha="center", va="bottom")

# (c) how each capture session contributes attacks
mix = (
    pd.crosstab(df["source_file"], df["attack_family"], normalize="index")
    .mul(100)
    .drop(columns=[c for c in ["Benign"] if c in df["attack_family"].unique()])
)
mix.plot(kind="bar", stacked=True, ax=axes[2], colormap="tab20", width=0.6)
axes[2].set_title("(c) Attack mix per source file (% of file)")
axes[2].set_ylabel("% of rows")
axes[2].tick_params(axis="x", rotation=0)
axes[2].legend(fontsize=7, loc="upper left")

plt.tight_layout()
plt.show()

print("Takeaways")
print("- Severe class imbalance -> accuracy is meaningless; use PR-AUC / recall / MCC.")
print("- Attack families are extremely unequal in size -> stratify every split.")
print("- Each capture session contains a different attack mix -> a random split")
print("  leaks session-specific traffic between train and test, so a")
print("  session-based generalisation test is run separately in Section 8.")

## 3.2 Data quality assessment

Flow-meter output has a well-known set of defects. Rather than assuming them,
each is measured:

* **Missing values** - `Flow Bytes/s` is undefined for zero-duration flows.
* **Infinities** - division by a zero duration yields `inf`, which silently
  breaks scalers, `LogisticRegression` and most metrics.
* **Constant / zero-variance columns** - several bulk-rate counters are always
  0 in this capture; they carry no information but cost memory and dilute
  feature-importance analysis.
* **Exact duplicate rows** - repeated identical flows inflate the apparent
  dataset size and, worse, put copies of the same flow into both the training
  and the test split, which inflates the reported score.
* **Negative values** - a negative duration or header length is physically
  impossible and indicates a capture artefact.

In [ ]:
# =============================================================================
# Quality report: one row per feature
# =============================================================================
META_COLS = [LABEL_COL, "source_file", "is_attack", "attack_family"]
raw_features = [c for c in df.columns if c not in META_COLS]
numeric = df[raw_features].select_dtypes(include=[np.number])
non_numeric = [c for c in raw_features if c not in numeric.columns]

quality = pd.DataFrame(
    {
        "dtype": numeric.dtypes.astype(str),
        "missing": numeric.isna().sum(),
        "infinite": np.isinf(numeric).sum(),
        "zeros": (numeric == 0).sum(),
        "negative": (numeric < 0).sum(),
        "n_unique": numeric.nunique(),
        "min": numeric.min(),
        "max": numeric.max(),
        "std": numeric.std(),
    }
)
quality["missing_%"] = 100 * quality["missing"] / len(df)
quality["zeros_%"] = 100 * quality["zeros"] / len(df)

constant_cols = quality.index[quality["n_unique"] <= 1].tolist()
inf_cols = quality.index[quality["infinite"] > 0].tolist()
na_cols = quality.index[quality["missing"] > 0].tolist()
neg_cols = quality.index[quality["negative"] > 0].tolist()

print(f"Numeric features        : {len(numeric.columns)}")
print(f"Non-numeric features    : {non_numeric if non_numeric else 'none'}")
print(f"Columns with missing    : {na_cols if na_cols else 'none'}")
print(f"Columns with +/-inf     : {inf_cols if inf_cols else 'none'}")
print(f"Constant columns        : {len(constant_cols)} -> {constant_cols}")
print(f"Columns with negatives  : {neg_cols if neg_cols else 'none'}")

t0 = time.perf_counter()
n_duplicates = int(df.duplicated(subset=raw_features + [LABEL_COL]).sum())
print(f"\nExact duplicate rows    : {n_duplicates:,} "
      f"({100 * n_duplicates / len(df):.2f}%)  [{time.perf_counter() - t0:.1f}s]")

print("\nFeatures with the highest share of zeros (candidates for removal)")
display(quality.sort_values("zeros_%", ascending=False).head(12)[
    ["n_unique", "zeros_%", "min", "max", "std"]
])

print("Features with the widest dynamic range (candidates for log transform)")
display(quality.assign(spread=quality["max"] - quality["min"])
        .sort_values("spread", ascending=False)
        .head(8)[["min", "max", "std"]])

## 3.3 Cleaning

The cleaning function is deliberately conservative and, importantly,
**leakage-free**:

| Step | Action | Rationale |
| --- | --- | --- |
| Infinities | `±inf → NaN` | keeps the row (the *fact* that a rate was undefined is informative) while making the value usable |
| Missing values | **left as `NaN`** here | imputation is done *inside* the model pipeline (`SimpleImputer`), so the median is learned from training folds only - imputing now would leak test information into training |
| Constant columns | dropped | zero information, non-zero cost |
| Duplicate rows | dropped | prevents the same flow appearing in train *and* test, which inflates scores |
| Precision | `float64 → float32` | halves memory, which is what makes a million-row table comfortable in Colab |
| Negative sentinels | clipped at 0 for physically non-negative counters | capture artefacts, not signal |

In [ ]:
# =============================================================================
# Cleaning (no imputation here - that happens inside the model pipelines)
# =============================================================================
def clean(frame: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    '''Apply the cleaning steps documented above and report what changed.'''
    out = frame.copy()
    before_rows, before_cols = out.shape
    before_mem = out.memory_usage(deep=True).sum() / 1024**3

    # 1. infinities -> NaN
    n_inf = int(np.isinf(out[feature_cols].to_numpy(dtype="float64", na_value=np.nan)).sum())
    out[feature_cols] = out[feature_cols].replace([np.inf, -np.inf], np.nan)

    # 2. drop zero-variance columns
    dropped_constant = [c for c in feature_cols if out[c].nunique(dropna=True) <= 1]
    out = out.drop(columns=dropped_constant)
    kept = [c for c in feature_cols if c not in dropped_constant]

    # 3. drop exact duplicates
    out = out.drop_duplicates(subset=kept + [LABEL_COL], keep="first").reset_index(drop=True)

    # 4. clip impossible negatives on non-negative counters
    clipped = [c for c in kept if (out[c] < 0).any()]
    if clipped:
        out[clipped] = out[clipped].clip(lower=0)

    # 5. downcast
    out[kept] = out[kept].astype("float32")

    after_mem = out.memory_usage(deep=True).sum() / 1024**3
    print(f"infinities converted to NaN : {n_inf:,}")
    print(f"constant columns dropped    : {len(dropped_constant)} {dropped_constant}")
    print(f"negative values clipped in   : {clipped if clipped else 'none'}")
    print(f"rows      {before_rows:>10,} -> {out.shape[0]:>10,} "
          f"({before_rows - out.shape[0]:,} duplicates removed)")
    print(f"columns   {before_cols:>10,} -> {out.shape[1]:>10,}")
    print(f"memory    {before_mem:>9.2f}G -> {after_mem:>9.2f}G")
    return out


df = clean(df, raw_features)
FEATURES_RAW = [c for c in df.columns if c not in META_COLS]

print(f"\nClean table: {df.shape[0]:,} rows x {len(FEATURES_RAW)} features")
print(f"Remaining NaNs (handled by the pipeline imputer): "
      f"{int(df[FEATURES_RAW].isna().sum().sum()):,}")
print(f"Class balance after de-duplication: "
      f"{100 * df['is_attack'].mean():.2f}% attack")

## 3.4 Exploratory analysis

Three questions drive the exploration, each aimed at a modelling decision:

1. **Are the features on comparable scales?** (decides whether scaling is
   required, and which model families are viable)
2. **Do benign and attack flows actually separate on individual features?**
   (indicates whether the problem is learnable and which features matter)
3. **How redundant is the feature set?** (drives the correlation pruning in
   Section 4)

In [ ]:
# =============================================================================
# 3.4a Scale and distribution of a representative feature subset
# =============================================================================
SPOTLIGHT = [
    c
    for c in [
        "Flow Duration", "Total Fwd Packets", "Total Backward Packets",
        "Flow Bytes/s", "Flow Packets/s", "Fwd Packet Length Mean",
        "Bwd Packet Length Mean", "Packet Length Std", "Average Packet Size",
        "Flow IAT Mean", "Init_Win_bytes_forward", "Destination Port",
    ]
    if c in df.columns
]

print("Descriptive statistics for representative features")
display(df[SPOTLIGHT].describe(percentiles=[0.01, 0.5, 0.99]).T)

print("Observation: ranges differ by 8+ orders of magnitude (ports ~10^2 vs")
print("byte rates ~10^9) and distributions are heavily right-skewed. Distance-")
print("and gradient-based models (k-NN, logistic regression) therefore need")
print("scaling; tree ensembles are scale invariant. Both are handled by using")
print("scikit-learn Pipelines with a per-model scaling switch.")

In [ ]:
# =============================================================================
# 3.4b Do benign and attack flows separate on single features?
# =============================================================================
eda_sample = df.sample(n=min(60_000, len(df)), random_state=RANDOM_STATE)
panel = [c for c in SPOTLIGHT if c != "Destination Port"][:6]

fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for ax, col in zip(axes.ravel(), panel):
    plot_df = eda_sample[[col, "is_attack"]].copy()
    # log1p purely for readability of the plot; the model sees raw values
    plot_df[col] = np.log1p(plot_df[col].clip(lower=0))
    sns.violinplot(
        data=plot_df, x="is_attack", y=col, hue="is_attack", ax=ax,
        palette={0: "#2a9d8f", 1: "#e63946"}, cut=0, legend=False,
    )
    ax.set_title(f"log1p({col})", fontsize=9)
    ax.set_xlabel("")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["benign", "attack"])
fig.suptitle(f"Feature distributions, benign vs attack "
             f"({len(eda_sample):,}-row random sample)", fontweight="bold")
plt.tight_layout()
plt.show()

# Quantify the separation with a rank-based effect size (Cliff's delta proxy:
# AUC of the single feature). Values far from 0.5 = individually informative.
single_auc = {}
for col in FEATURES_RAW:
    values = eda_sample[col].to_numpy()
    if np.isnan(values).all():
        continue
    ranks = pd.Series(values).rank(na_option="bottom").to_numpy()
    y_s = eda_sample["is_attack"].to_numpy()
    n1, n0 = y_s.sum(), len(y_s) - y_s.sum()
    if n1 == 0 or n0 == 0:
        continue
    single_auc[col] = (ranks[y_s == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

auc_series = pd.Series(single_auc).sub(0.5).abs().sort_values(ascending=False)
print("Most individually discriminative raw features (|univariate AUC - 0.5|)")
display(auc_series.head(15).to_frame("separation"))

In [ ]:
# =============================================================================
# 3.4c Feature redundancy and destination-port behaviour
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6),
                         gridspec_kw={"width_ratios": [1.15, 1]})

# --- correlation structure of the 28 highest-variance features ---------------
corr_sample = df.sample(n=min(120_000, len(df)), random_state=RANDOM_STATE)
top_var = (
    corr_sample[FEATURES_RAW].std().sort_values(ascending=False).head(28).index.tolist()
)
corr = corr_sample[top_var].corr().abs()
sns.heatmap(corr, cmap="rocket_r", vmin=0, vmax=1, ax=axes[0],
            xticklabels=True, yticklabels=True, cbar_kws={"shrink": 0.7})
axes[0].set_title("|Pearson correlation|, 28 highest-variance features")
axes[0].tick_params(labelsize=6)

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
redundant_pairs = (
    upper.stack().sort_values(ascending=False).head(10).rename("|r|").reset_index()
)
redundant_pairs.columns = ["feature_a", "feature_b", "|r|"]

# --- attack rate by destination port ----------------------------------------
if "Destination Port" in df.columns:
    port_stats = (
        df.groupby(df["Destination Port"].astype("int64"))
        .agg(flows=("is_attack", "size"), attack_rate=("is_attack", "mean"))
        .query("flows >= 500")
        .sort_values("flows", ascending=False)
        .head(15)
        .sort_values("attack_rate")
    )
    axes[1].barh(port_stats.index.astype(str), 100 * port_stats["attack_rate"],
                 color="#e63946")
    axes[1].set_title("Attack rate by destination port (busiest 15 ports)")
    axes[1].set_xlabel("% of flows that are attacks")
    axes[1].set_ylabel("destination port")
    for y, (v, n) in enumerate(zip(port_stats["attack_rate"], port_stats["flows"])):
        axes[1].text(100 * v + 1, y, f"n={n:,}", va="center", fontsize=7)

plt.tight_layout()
plt.show()

print("Most redundant feature pairs (near-duplicate information)")
display(redundant_pairs)
print("These are pruned automatically in Section 4 at |r| > "
      f"{CFG.corr_drop_threshold}.")
print("\nPort behaviour is strongly predictive in this capture, but that is")
print("partly an artefact of the lab topology (attacks were launched at specific")
print("services). This is revisited as a generalisation risk in Section 8 and")
print("as a fairness/validity risk in Section 9.")

---
# Section 4 - Feature engineering

Raw flow counters are *absolute* quantities, so they conflate "what kind of
conversation is this" with "how long did it last". The engineered features below
are mostly **scale-invariant ratios and rates**, which is exactly what
distinguishes attack traffic from normal traffic:

| # | Engineered feature(s) | Detection intuition |
| --- | --- | --- |
| 1 | `duration_s`, `log_duration` | tames an 11-order-of-magnitude range; slow-rate DoS lives in the long tail |
| 2 | `fwd_bwd_packet_ratio`, `fwd_bwd_byte_ratio` | scans and floods are one-directional; a browsing session is balanced |
| 3 | `flow_symmetry` | single bounded score for the same idea, robust to zero backward traffic |
| 4 | `bytes_per_packet`, `fwd_bytes_per_packet`, `bwd_bytes_per_packet` | SYN/port scans send tiny packets; exfiltration sends full-MTU packets |
| 5 | `packets_per_second`, `bytes_per_second` | recomputed cleanly (the meter's own rate columns are the `inf`/`NaN` source) |
| 6 | `packet_len_cv`, `fwd_len_cv` | automated traffic is uniform (low coefficient of variation); humans are erratic |
| 7 | `iat_burstiness`, `iat_range_ratio` | inter-arrival regularity is a strong bot/C2 beaconing signal |
| 8 | `total_flags`, `flag_density`, `syn_ratio`, `rst_ratio`, `psh_ratio` | flag *composition* separates SYN floods and RST scans from real sessions |
| 9 | `header_overhead` | high header-to-payload ratio implies many empty control packets |
| 10 | `active_idle_ratio`, `idle_share` | slowloris-style attacks hold connections open and mostly idle |
| 11 | `port_class`, `port_is_web/dns/mail/remote`, `port_is_ephemeral` | turns an arbitrary integer into meaningful service semantics; a raw port number is a *nominal* value that trees would otherwise split on numerically |
| 12 | `log1p(...)` of six volume counters | gives the linear models a fighting chance against heavy skew |

Two engineering rules are enforced throughout:

* **Safe arithmetic.** Every division goes through `_safe_div`, so no engineered
  column can reintroduce `inf`.
* **Defensive column access.** `_col` returns `NaN` for an absent column, so the
  notebook still runs if an export is missing a counter, instead of dying with a
  `KeyError` half way through.

Finally, near-duplicate features (`|r| > 0.98`) are pruned. This is done
*after* engineering, so a redundant raw counter can be replaced by the more
informative ratio derived from it.

In [ ]:
# =============================================================================
# Feature engineering
# =============================================================================
EPS = 1e-6

WELL_KNOWN_PORTS = {
    "web": (80, 443, 8080, 8443),
    "dns": (53,),
    "mail": (25, 110, 143, 465, 587, 993, 995),
    "remote": (22, 23, 3389, 5900),
    "file": (20, 21, 139, 445, 2049),
    "directory": (88, 389, 636, 464),
}


def _col(frame: pd.DataFrame, name: str) -> pd.Series:
    '''Return a column as float32, or an all-NaN column if it is absent.'''
    if name in frame.columns:
        return frame[name].astype("float32")
    return pd.Series(np.nan, index=frame.index, dtype="float32")


def _safe_div(num: pd.Series, den: pd.Series) -> pd.Series:
    '''Division that can never produce inf: zero denominators -> NaN.'''
    den = den.astype("float32")
    out = num.astype("float32") / den.where(den.abs() > EPS)
    return out.replace([np.inf, -np.inf], np.nan).astype("float32")


def engineer_features(frame: pd.DataFrame) -> pd.DataFrame:
    '''Add the engineered features documented in the table above.'''
    out = frame.copy()

    fwd_pkts = _col(out, "Total Fwd Packets")
    bwd_pkts = _col(out, "Total Backward Packets")
    fwd_bytes = _col(out, "Total Length of Fwd Packets")
    bwd_bytes = _col(out, "Total Length of Bwd Packets")
    total_pkts = fwd_pkts.add(bwd_pkts, fill_value=0)
    total_bytes = fwd_bytes.add(bwd_bytes, fill_value=0)

    # 1. duration -----------------------------------------------------------
    duration_us = _col(out, "Flow Duration")
    out["duration_s"] = duration_us / 1e6
    out["log_duration"] = np.log1p(duration_us.clip(lower=0))

    # 2-3. directionality ---------------------------------------------------
    out["fwd_bwd_packet_ratio"] = _safe_div(fwd_pkts, bwd_pkts + 1)
    out["fwd_bwd_byte_ratio"] = _safe_div(fwd_bytes, bwd_bytes + 1)
    out["flow_symmetry"] = _safe_div((fwd_pkts - bwd_pkts).abs(), total_pkts)
    out["total_packets"] = total_pkts
    out["total_bytes"] = total_bytes

    # 4. payload shape ------------------------------------------------------
    out["bytes_per_packet"] = _safe_div(total_bytes, total_pkts)
    out["fwd_bytes_per_packet"] = _safe_div(fwd_bytes, fwd_pkts)
    out["bwd_bytes_per_packet"] = _safe_div(bwd_bytes, bwd_pkts)

    # 5. clean rates --------------------------------------------------------
    out["packets_per_second"] = _safe_div(total_pkts, out["duration_s"])
    out["bytes_per_second"] = _safe_div(total_bytes, out["duration_s"])

    # 6. regularity of packet sizes ----------------------------------------
    out["packet_len_cv"] = _safe_div(_col(out, "Packet Length Std"),
                                     _col(out, "Packet Length Mean"))
    out["fwd_len_cv"] = _safe_div(_col(out, "Fwd Packet Length Std"),
                                  _col(out, "Fwd Packet Length Mean"))
    out["bwd_len_cv"] = _safe_div(_col(out, "Bwd Packet Length Std"),
                                  _col(out, "Bwd Packet Length Mean"))

    # 7. timing regularity --------------------------------------------------
    out["iat_burstiness"] = _safe_div(_col(out, "Flow IAT Std"),
                                      _col(out, "Flow IAT Mean"))
    out["iat_range_ratio"] = _safe_div(
        _col(out, "Flow IAT Max") - _col(out, "Flow IAT Min"),
        _col(out, "Flow IAT Mean"),
    )

    # 8. TCP flag composition ----------------------------------------------
    flag_cols = [
        c for c in [
            "FIN Flag Count", "SYN Flag Count", "RST Flag Count", "PSH Flag Count",
            "ACK Flag Count", "URG Flag Count", "CWE Flag Count", "ECE Flag Count",
        ] if c in out.columns
    ]
    if flag_cols:
        out["total_flags"] = out[flag_cols].sum(axis=1).astype("float32")
        out["flag_density"] = _safe_div(out["total_flags"], total_pkts)
        out["syn_ratio"] = _safe_div(_col(out, "SYN Flag Count"), out["total_flags"] + 1)
        out["rst_ratio"] = _safe_div(_col(out, "RST Flag Count"), out["total_flags"] + 1)
        out["psh_ratio"] = _safe_div(_col(out, "PSH Flag Count"), out["total_flags"] + 1)

    # 9. protocol overhead --------------------------------------------------
    header_bytes = _col(out, "Fwd Header Length").add(
        _col(out, "Bwd Header Length"), fill_value=0
    )
    out["header_overhead"] = _safe_div(header_bytes, total_bytes + 1)

    # 10. active / idle behaviour ------------------------------------------
    active_mean = _col(out, "Active Mean")
    idle_mean = _col(out, "Idle Mean")
    out["active_idle_ratio"] = _safe_div(active_mean, idle_mean + 1)
    out["idle_share"] = _safe_div(idle_mean, active_mean + idle_mean + 1)

    # 11. service semantics from the destination port ----------------------
    if "Destination Port" in out.columns:
        port = _col(out, "Destination Port").fillna(-1).astype("int32")
        out["port_class"] = np.select(
            [port < 1024, port < 49152], [0, 1], default=2
        ).astype("float32")
        out["port_is_ephemeral"] = (port >= 49152).astype("float32")
        for service, ports in WELL_KNOWN_PORTS.items():
            out[f"port_is_{service}"] = port.isin(ports).astype("float32")

    # 12. log transforms of skewed volume counters -------------------------
    for col in ["Total Length of Fwd Packets", "Total Length of Bwd Packets",
                "Flow IAT Mean", "Flow IAT Max", "Active Mean", "Idle Mean"]:
        if col in out.columns:
            key = "log_" + col.lower().replace(" ", "_").replace("/", "_per_")
            out[key] = np.log1p(_col(out, col).clip(lower=0))

    engineered = [c for c in out.columns if c not in frame.columns]
    out[engineered] = out[engineered].replace([np.inf, -np.inf], np.nan).astype("float32")
    return out


t0 = time.perf_counter()
df = engineer_features(df)
ENGINEERED = [c for c in df.columns if c not in META_COLS + FEATURES_RAW]
print(f"Engineered {len(ENGINEERED)} new features in "
      f"{time.perf_counter() - t0:.1f}s\n")
print("\n".join(f"  {i:>2}. {c}" for i, c in enumerate(ENGINEERED, 1)))

assert not np.isinf(df[ENGINEERED].to_numpy(dtype="float32", na_value=0.0)).any(), \
    "engineered features must never contain infinities"
print(f"\nInfinity check passed. Table is now "
      f"{df.shape[0]:,} rows x {len(FEATURES_RAW) + len(ENGINEERED)} features.")

In [ ]:
# =============================================================================
# Prune near-duplicate features, then rank what survived
# =============================================================================
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer

candidate_features = FEATURES_RAW + ENGINEERED

prune_sample = df.sample(n=min(200_000, len(df)), random_state=RANDOM_STATE)
corr_matrix = prune_sample[candidate_features].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
DROPPED_CORRELATED = [
    col for col in upper.columns if (upper[col] > CFG.corr_drop_threshold).any()
]
FEATURES = [c for c in candidate_features if c not in DROPPED_CORRELATED]

print(f"Candidate features            : {len(candidate_features)}")
print(f"Dropped at |r| > {CFG.corr_drop_threshold}        : {len(DROPPED_CORRELATED)}")
print(f"Final modelling feature set   : {len(FEATURES)}")
print(f"  raw counters kept           : {len([c for c in FEATURES if c in FEATURES_RAW])}")
print(f"  engineered kept             : {len([c for c in FEATURES if c in ENGINEERED])}")
print("\nDropped as redundant:")
print("  " + ", ".join(DROPPED_CORRELATED) if DROPPED_CORRELATED else "  none")

# --- mutual information: model-free, non-linear relevance ranking -----------
mi_sample = df.sample(n=min(25_000, len(df)), random_state=RANDOM_STATE)
mi_x = SimpleImputer(strategy="median").fit_transform(mi_sample[FEATURES])
t0 = time.perf_counter()
mi_scores = pd.Series(
    mutual_info_classif(mi_x, mi_sample["is_attack"], random_state=RANDOM_STATE,
                        n_neighbors=3),
    index=FEATURES,
).sort_values(ascending=False)
print(f"\nMutual information computed on {len(mi_sample):,} rows in "
      f"{time.perf_counter() - t0:.1f}s")

fig, ax = plt.subplots(figsize=(9, 6))
top_mi = mi_scores.head(22).sort_values()
bar_colours = ["#e63946" if f in ENGINEERED else "#457b9d" for f in top_mi.index]
ax.barh(top_mi.index, top_mi.values, color=bar_colours)
ax.set_title("Top 22 features by mutual information with is_attack")
ax.set_xlabel("mutual information (nats)")
handles = [
    plt.Rectangle((0, 0), 1, 1, color="#e63946"),
    plt.Rectangle((0, 0), 1, 1, color="#457b9d"),
]
ax.legend(handles, ["engineered", "raw"], loc="lower right")
plt.tight_layout()
plt.show()

n_eng_in_top20 = sum(f in ENGINEERED for f in mi_scores.head(20).index)
print(f"{n_eng_in_top20} of the 20 most informative features are engineered,")
print("which is the evidence that the feature engineering added signal rather")
print("than just widening the table.")

## 4.1 Data splitting strategy

| Decision | Choice | Why |
| --- | --- | --- |
| Split type | stratified hold-out (75 / 25) | the rarest attack family has only a handful of rows; an unstratified split can leave zero of them in the test set |
| Stratification key | the **fine-grained label**, not the binary target | preserves the mix *within* the attack class, so per-family recall is measurable |
| Sample size | `model_sample` rows, stratified | full data is ~10^6 x ~100; the randomised searches would take hours on a free Colab CPU. Stratified sampling preserves every class proportion, so the estimates stay unbiased - only their variance grows slightly. Raise `CFG.model_sample` to use more data |
| Selection vs evaluation | model selection and tuning use **cross-validation on the training split only**; the test split is touched once, in Section 6 | prevents the optimistic bias of selecting a model on the same data used to report its score |
| Imputation & scaling | inside `Pipeline` | fitted per fold on training data only - no leakage |

A `meta_test` frame keeps the label, family, source file and destination port
for the test rows. It is used for per-family recall and for the fairness audit,
and is never given to a model as input.

In [ ]:
# =============================================================================
# Stratified sampling and train / test split
# =============================================================================
from sklearn.model_selection import train_test_split

strata = df[LABEL_COL].astype(str)
# Classes too small to stratify safely are pooled into one bucket.
rare = strata.value_counts()[lambda s: s < 10].index
strata = strata.where(~strata.isin(rare), "RARE_POOLED")

if len(df) > CFG.model_sample:
    keep_idx, _ = train_test_split(
        df.index, train_size=CFG.model_sample, stratify=strata,
        random_state=RANDOM_STATE,
    )
    work = df.loc[keep_idx].reset_index(drop=True)
    print(f"Stratified sample: {len(work):,} of {len(df):,} rows "
          f"({100 * len(work) / len(df):.1f}%)")
else:
    work = df.reset_index(drop=True)
    print(f"Using all {len(work):,} rows")

work_strata = work[LABEL_COL].astype(str)
work_strata = work_strata.where(
    ~work_strata.isin(work_strata.value_counts()[lambda s: s < 10].index),
    "RARE_POOLED",
)

X = work[FEATURES]
y = work["is_attack"].astype("int8")

X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
    X, y, work[[LABEL_COL, "attack_family", "source_file"]
               + (["Destination Port"] if "Destination Port" in work.columns else [])],
    test_size=CFG.test_size, stratify=work_strata, random_state=RANDOM_STATE,
)

# Smaller stratified slice of the *training* data for model selection & tuning.
cv_rows = min(CFG.cv_sample, len(X_train))
X_cv, _, y_cv = X_train, None, y_train
if cv_rows < len(X_train):
    X_cv, _, y_cv, _ = train_test_split(
        X_train, y_train, train_size=cv_rows, stratify=y_train,
        random_state=RANDOM_STATE,
    )

summary = pd.DataFrame(
    {
        "rows": [len(X_train), len(X_test), len(X_cv)],
        "attack rows": [int(y_train.sum()), int(y_test.sum()), int(y_cv.sum())],
        "attack %": [100 * y_train.mean(), 100 * y_test.mean(), 100 * y_cv.mean()],
    },
    index=["train (fit)", "test (held out)", "cv slice (selection/tuning)"],
)
print()
display(summary)

print("Attack families present in the test split (per-family recall is reported "
      "in Section 6):")
display(meta_test["attack_family"].value_counts().to_frame("test rows"))

---
# Section 5 - Model selection

## Candidate shortlist and why each one is here

Model choice is driven by the shape of the problem: ~100 numeric features,
heavily skewed and non-linearly related to the target, severe class imbalance,
a million-row training set, and a hard requirement for fast inference.

| Candidate | Family | Reason for inclusion | Expected weakness |
| --- | --- | --- | --- |
| `DummyClassifier` | baseline | Establishes the floor. With ~30% attacks, "always benign" already scores high *accuracy* - this is the cell that proves accuracy must not be used | no learning at all |
| `LogisticRegression` | linear | Fast, calibrated, fully interpretable coefficients; the benchmark a complex model must beat to justify itself | cannot express feature interactions or thresholds |
| `GaussianNB` | probabilistic | Extremely cheap, useful sanity check | its independence and normality assumptions are badly violated here |
| `DecisionTree` | single tree | Human-readable rules, captures thresholds; shows how much of the performance comes from a *single* tree | high variance, overfits |
| `k-NN (k=5)` | instance based | Strong on locally-clustered attack traffic; no training cost | prediction is O(n) per flow - likely disqualifying for inline use |
| `RandomForest` | bagged trees | Robust default for tabular data, handles mixed scales, gives importances, parallelises | larger memory, slower scoring than boosting |
| `ExtraTrees` | bagged trees | More randomised splits, often better variance reduction and faster to fit | slightly weaker on very noisy features |
| `HistGradientBoosting` | boosted trees | State of the art for tabular data; histogram binning makes it fast on 10^5-10^6 rows and it handles `NaN` natively | sequential fitting, more hyperparameters |

Deliberately excluded: SVM with an RBF kernel (training is roughly quadratic -
impractical at this scale) and deep neural networks (no accuracy advantage on
tabular features of this kind, at a large cost in tuning, latency and
explainability).

## Protocol

* **`Pipeline` for every candidate** - `SimpleImputer(median)` always,
  `StandardScaler` only for the scale-sensitive models. This guarantees the
  imputation median and the scaling statistics are learned inside each training
  fold, never from the validation fold.
* **`StratifiedKFold(5, shuffle=True)`** - preserves the imbalance in every fold.
* **`class_weight="balanced"`** where the estimator supports it, so the minority
  class is not ignored.
* **Multiple metrics at once**, plus fit and score time, because a model that
  wins on PR-AUC but takes 20 ms per flow is not deployable.
* Run on the `cv slice` of the *training* data only - the test split stays sealed
  until Section 6.

In [ ]:
# =============================================================================
# 5.1 Candidate models, each wrapped in a leakage-free pipeline
# =============================================================================
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier


def make_pipeline(estimator, scale: bool = False) -> Pipeline:
    '''Impute (always) and scale (only when the estimator needs it).'''
    steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scale", StandardScaler()))
    steps.append(("model", estimator))
    return Pipeline(steps)


CANDIDATES: dict[str, dict] = {
    "Dummy (majority class)": {
        "estimator": DummyClassifier(strategy="most_frequent"),
        "scale": False,
    },
    "Logistic Regression": {
        "estimator": LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE,
        ),
        "scale": True,
    },
    "Gaussian Naive Bayes": {"estimator": GaussianNB(), "scale": True},
    "Decision Tree": {
        "estimator": DecisionTreeClassifier(
            max_depth=18, min_samples_leaf=5, class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        "scale": False,
    },
    "k-NN (k=5)": {
        "estimator": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "scale": True,
        "max_rows": CFG.knn_sample,   # O(n) scoring: keep the sample small
    },
    "Random Forest": {
        "estimator": RandomForestClassifier(
            n_estimators=120, min_samples_leaf=2, class_weight="balanced_subsample",
            n_jobs=-1, random_state=RANDOM_STATE,
        ),
        "scale": False,
    },
    "Extra Trees": {
        "estimator": ExtraTreesClassifier(
            n_estimators=120, min_samples_leaf=2, class_weight="balanced_subsample",
            n_jobs=-1, random_state=RANDOM_STATE,
        ),
        "scale": False,
    },
    "HistGradientBoosting": {
        "estimator": HistGradientBoostingClassifier(
            max_iter=200, learning_rate=0.1, random_state=RANDOM_STATE,
        ),
        "scale": False,
    },
}

SCORING = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "f1": "f1",
    "recall": "recall",
    "precision": "precision",
    "mcc": "matthews_corrcoef",
    "balanced_accuracy": "balanced_accuracy",
    "accuracy": "accuracy",
}

print(f"{len(CANDIDATES)} candidates x {CFG.cv_folds} folds, "
      f"{len(SCORING)} metrics each.")
print(f"Cross-validation slice: {len(X_cv):,} rows x {len(FEATURES)} features.")

In [ ]:
# =============================================================================
# 5.2 Cross-validated comparison  (the slowest cell in Section 5, ~3-6 min)
# =============================================================================
cv_splitter = StratifiedKFold(
    n_splits=CFG.cv_folds, shuffle=True, random_state=RANDOM_STATE
)
selection_rows = []

for name, spec in CANDIDATES.items():
    limit = spec.get("max_rows", len(X_cv))
    if limit < len(X_cv):
        X_use, _, y_use, _ = train_test_split(
            X_cv, y_cv, train_size=limit, stratify=y_cv, random_state=RANDOM_STATE
        )
        note = f"{limit:,} rows"
    else:
        X_use, y_use, note = X_cv, y_cv, f"{len(X_cv):,} rows"

    t0 = time.perf_counter()
    scores = cross_validate(
        make_pipeline(spec["estimator"], spec["scale"]),
        X_use, y_use, cv=cv_splitter, scoring=SCORING, n_jobs=1,
        return_train_score=True, error_score="raise",
    )
    elapsed = time.perf_counter() - t0

    row = {"model": name, "rows used": note}
    for metric in SCORING:
        row[metric] = scores[f"test_{metric}"].mean()
        row[f"{metric}_sd"] = scores[f"test_{metric}"].std()
    row["overfit gap (f1)"] = (
        scores["train_f1"].mean() - scores["test_f1"].mean()
    )
    row["fit s/fold"] = scores["fit_time"].mean()
    row["score s/fold"] = scores["score_time"].mean()
    row["total s"] = elapsed
    selection_rows.append(row)
    print(f"{name:<24} PR-AUC={row['pr_auc']:.4f}  F1={row['f1']:.4f}  "
          f"recall={row['recall']:.4f}  ({elapsed:5.1f}s, {note})")

selection = (
    pd.DataFrame(selection_rows).set_index("model").sort_values("pr_auc", ascending=False)
)

print("\nCross-validated model comparison (mean over folds, sorted by PR-AUC)")
display(
    selection[["rows used"] + list(SCORING)]
    .style.format(dict.fromkeys(SCORING, "{:.4f}"))
    .background_gradient(subset=["pr_auc", "f1", "mcc"], cmap="Greens")
)

print("Stability and cost")
display(selection[["pr_auc_sd", "f1_sd", "overfit gap (f1)", "fit s/fold",
                   "score s/fold"]])

RESULTS.update(
    {f"CV: {name}": {"pr_auc": row["pr_auc"], "f1": row["f1"],
                     "recall": row["recall"], "precision": row["precision"],
                     "stage": "model selection (5-fold CV)"}
     for name, row in selection.iterrows()}
)

In [ ]:
# =============================================================================
# 5.3 Visual comparison: quality, stability and cost
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
ordered_models = selection.index.tolist()[::-1]

# (a) quality with fold-to-fold error bars
y_pos = np.arange(len(ordered_models))
for offset, (metric, colour) in enumerate(
    [("pr_auc", "#1d3557"), ("f1", "#457b9d"), ("recall", "#e63946")]
):
    axes[0].barh(
        y_pos + (offset - 1) * 0.26,
        selection.loc[ordered_models, metric],
        xerr=selection.loc[ordered_models, f"{metric}_sd"],
        height=0.26, label=metric.upper(), color=colour, error_kw={"lw": 0.8},
    )
axes[0].set_yticks(y_pos, ordered_models, fontsize=8)
axes[0].set_xlim(0, 1.02)
axes[0].set_title("(a) Quality (error bars = SD across folds)")
axes[0].legend(fontsize=8, loc="lower right")

# (b) accuracy is misleading - show it against MCC
axes[1].barh(y_pos - 0.2, selection.loc[ordered_models, "accuracy"], height=0.4,
             color="#adb5bd", label="accuracy")
axes[1].barh(y_pos + 0.2, selection.loc[ordered_models, "mcc"], height=0.4,
             color="#2a9d8f", label="MCC")
axes[1].set_yticks(y_pos, ordered_models, fontsize=8)
axes[1].axvline(1 - y_cv.mean(), ls="--", c="k", lw=1)
axes[1].text(1 - y_cv.mean(), len(ordered_models) - 0.4,
             " accuracy of\n always-benign", fontsize=7, va="top")
axes[1].set_title("(b) Why accuracy is the wrong metric")
axes[1].legend(fontsize=8, loc="lower right")

# (c) cost / benefit
for name in selection.index:
    axes[2].scatter(selection.loc[name, "score s/fold"],
                    selection.loc[name, "pr_auc"], s=70)
    axes[2].annotate(name, (selection.loc[name, "score s/fold"],
                            selection.loc[name, "pr_auc"]),
                     fontsize=7, xytext=(4, 4), textcoords="offset points")
axes[2].set_xscale("log")
axes[2].set_xlabel("scoring time per fold (s, log scale)")
axes[2].set_ylabel("PR-AUC")
axes[2].set_title("(c) Detection quality vs inference cost")

plt.tight_layout()
plt.show()

best_by_pr = selection.index[0]
dummy_acc = selection.loc["Dummy (majority class)", "accuracy"]
print(f"The trivial always-benign classifier already reaches "
      f"{dummy_acc:.1%} accuracy but MCC = "
      f"{selection.loc['Dummy (majority class)', 'mcc']:.3f} and recall = 0:")
print("it detects nothing. Accuracy is therefore excluded from all further")
print("model decisions in favour of PR-AUC, recall and MCC.\n")
print(f"Best candidate by PR-AUC: {best_by_pr}")

## Selection decision

Two models are carried forward to tuning and full evaluation:

* **`HistGradientBoosting`** - the accuracy leader on the cross-validated
  ranking, with the fastest scoring time of the strong models (histogram binning
  keeps inference cheap), native `NaN` handling and a small serialised size.
* **`RandomForest`** - kept as a robust counterweight. It needs almost no tuning
  to work, its per-tree structure supports straightforward importance and
  path-based explanations, and having a second architecture guards against the
  selection being an artefact of one algorithm's inductive bias.

`k-NN` is rejected on operational grounds regardless of its score: panel (c)
shows its scoring cost is orders of magnitude above the tree models, because it
must compare each new flow against the entire training set. `GaussianNB` and
`LogisticRegression` are rejected on quality - the gap to the ensembles is the
empirical evidence that the decision boundary is strongly non-linear.

---
# Section 6 - Performance measurement

## Choosing the right metrics

With ~30% attacks and a heavily asymmetric cost of error, the metric set is
chosen deliberately:

| Metric | What it answers | Why it is (or is not) trusted here |
| --- | --- | --- |
| **Accuracy** | share of correct predictions | **Not trusted.** Dominated by the majority class - see Section 5, panel (b) |
| **Recall (attack)** | what fraction of real attacks did we catch? | The primary safety metric: a false negative is an undetected intrusion |
| **Precision (attack)** | when we alert, how often is it real? | The primary cost metric: false positives consume analyst time and may block paying customers |
| **F1** | balance of the two | Single comparable number, but it hides *which* side is failing |
| **PR-AUC** | ranking quality on the minority class across all thresholds | Best threshold-free summary under imbalance; unlike ROC-AUC it is not flattered by a large benign class |
| **ROC-AUC** | separability | Reported for comparability with published work, interpreted cautiously |
| **MCC** | correlation over the whole confusion matrix | Only metric that degrades if *any* of the four cells is bad; good single-number honesty check |
| **Balanced accuracy** | mean of per-class recall | Treats a rare attack family as importantly as bulk benign traffic |
| **Expected cost** | business loss per 1,000 flows | Converts the confusion matrix into the units the client actually cares about |

Both shortlisted models are now fit on the **full training split** and evaluated
**once** on the sealed test split.

In [ ]:
# =============================================================================
# 6.1 Fit the shortlisted models on the full training split and evaluate once
# =============================================================================
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)


def evaluate(name: str, model, X_eval, y_eval, threshold: float = 0.5,
             stage: str = "test", store: bool = True) -> dict:
    '''Score a fitted model and (optionally) record it in the RESULTS registry.'''
    proba = model.predict_proba(X_eval)[:, 1]
    pred = (proba >= threshold).astype("int8")
    tn, fp, fn, tp = confusion_matrix(y_eval, pred, labels=[0, 1]).ravel()
    metrics = {
        "threshold": threshold,
        "pr_auc": average_precision_score(y_eval, proba),
        "roc_auc": roc_auc_score(y_eval, proba),
        "f1": f1_score(y_eval, pred, zero_division=0),
        "recall": recall_score(y_eval, pred, zero_division=0),
        "precision": precision_score(y_eval, pred, zero_division=0),
        "mcc": matthews_corrcoef(y_eval, pred),
        "balanced_accuracy": balanced_accuracy_score(y_eval, pred),
        "brier": brier_score_loss(y_eval, proba),
        "false_positive_rate": fp / max(fp + tn, 1),
        "false_negative_rate": fn / max(fn + tp, 1),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
        "cost_per_1k_flows": 1000
        * (CFG.cost_false_negative * fn + CFG.cost_false_positive * fp)
        / len(y_eval),
        "stage": stage,
    }
    if store:
        RESULTS[name] = metrics
    return metrics


FINALISTS = {
    "Random Forest": CANDIDATES["Random Forest"],
    "HistGradientBoosting": CANDIDATES["HistGradientBoosting"],
}

fitted: dict[str, Pipeline] = {}
baseline_rows = []
for name, spec in FINALISTS.items():
    pipe = make_pipeline(spec["estimator"], spec["scale"])
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    fit_s = time.perf_counter() - t0
    fitted[name] = pipe
    row = evaluate(f"{name} (default)", pipe, X_test, y_test,
                   stage="test / default hyperparameters")
    row["fit_seconds"] = fit_s
    baseline_rows.append(pd.Series(row, name=name))
    print(f"{name:<22} fitted on {len(X_train):,} rows in {fit_s:5.1f}s")

baseline = pd.DataFrame(baseline_rows)
# "stage" is text; everything else must be numeric for sorting and plotting.
numeric_baseline = [c for c in baseline.columns if c != "stage"]
baseline[numeric_baseline] = baseline[numeric_baseline].astype(float)

print("\nHeld-out test performance at the default 0.50 threshold")
display(baseline[["pr_auc", "roc_auc", "f1", "recall", "precision", "mcc",
                  "balanced_accuracy", "false_positive_rate",
                  "cost_per_1k_flows", "fit_seconds"]])

print("Confusion-matrix counts")
display(baseline[["tp", "fp", "fn", "tn"]])

CHAMPION = baseline["pr_auc"].idxmax()
print(f"Champion on held-out PR-AUC: {CHAMPION}\n")
print(f"Detailed classification report - {CHAMPION}")
print(classification_report(
    y_test, fitted[CHAMPION].predict(X_test),
    target_names=["benign", "attack"], digits=4,
))

In [ ]:
# =============================================================================
# 6.2 Confusion matrices, ROC and precision-recall curves
# =============================================================================
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

for ax, (name, model) in zip(axes[0], fitted.items()):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test, display_labels=["benign", "attack"],
        normalize="true", cmap="Blues", values_format=".4f", ax=ax, colorbar=False,
    )
    ax.set_title(f"{name}\nrow-normalised (recall on the diagonal)", fontsize=10)

for name, model in fitted.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=axes[1][0], name=name)
    PrecisionRecallDisplay.from_estimator(model, X_test, y_test, ax=axes[1][1],
                                          name=name)

axes[1][0].plot([0, 1], [0, 1], "k--", lw=1, label="random")
axes[1][0].set_title("ROC curves")
axes[1][0].legend(fontsize=8)
axes[1][1].axhline(y_test.mean(), ls="--", c="k", lw=1,
                   label=f"no-skill = {y_test.mean():.3f}")
axes[1][1].set_title("Precision-recall curves (the honest view under imbalance)")
axes[1][1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print("The ROC curves are almost indistinguishable and both hug the top-left")
print("corner - typical, and the reason ROC-AUC is a weak discriminator under")
print("imbalance. The PR curves separate the models much more clearly, which is")
print("why PR-AUC is the headline metric and the tuning objective in Section 7.")

In [ ]:
# =============================================================================
# 6.3 Threshold selection: the metric the business actually pays for
# =============================================================================
proba_champion = fitted[CHAMPION].predict_proba(X_test)[:, 1]

grid = np.round(np.linspace(0.01, 0.99, 99), 2)
sweep = []
for thr in grid:
    pred = (proba_champion >= thr).astype("int8")
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    sweep.append(
        {
            "threshold": thr,
            "recall": tp / max(tp + fn, 1),
            "precision": tp / max(tp + fp, 1),
            "f1": f1_score(y_test, pred, zero_division=0),
            "alerts_per_1k": 1000 * (tp + fp) / len(y_test),
            "missed_per_1k": 1000 * fn / len(y_test),
            "cost_per_1k": 1000
            * (CFG.cost_false_negative * fn + CFG.cost_false_positive * fp)
            / len(y_test),
        }
    )
sweep = pd.DataFrame(sweep)

thr_cost = float(sweep.loc[sweep["cost_per_1k"].idxmin(), "threshold"])
thr_f1 = float(sweep.loc[sweep["f1"].idxmax(), "threshold"])
high_precision = sweep.query("precision >= 0.99")
thr_p99 = float(high_precision["threshold"].min()) if len(high_precision) else np.nan

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
axes[0].plot(sweep["threshold"], sweep["precision"], label="precision")
axes[0].plot(sweep["threshold"], sweep["recall"], label="recall")
axes[0].plot(sweep["threshold"], sweep["f1"], label="F1")
axes[0].axvline(thr_cost, c="#e63946", ls="--",
                label=f"min-cost = {thr_cost:.2f}")
axes[0].set_xlabel("decision threshold")
axes[0].set_title("(a) Precision / recall trade-off")
axes[0].legend(fontsize=8)

axes[1].plot(sweep["threshold"], sweep["cost_per_1k"], c="#1d3557")
axes[1].axvline(thr_cost, c="#e63946", ls="--")
axes[1].set_xlabel("decision threshold")
axes[1].set_ylabel(f"cost per 1,000 flows\n(FN={CFG.cost_false_negative:g}, "
                   f"FP={CFG.cost_false_positive:g})")
axes[1].set_title("(b) Expected business cost")

axes[2].plot(sweep["threshold"], sweep["alerts_per_1k"], label="alerts raised")
axes[2].plot(sweep["threshold"], sweep["missed_per_1k"], label="attacks missed")
axes[2].axvline(thr_cost, c="#e63946", ls="--")
axes[2].set_xlabel("decision threshold")
axes[2].set_ylabel("per 1,000 flows")
axes[2].set_title("(c) SOC workload vs risk")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"Cost model: a missed attack costs {CFG.cost_false_negative:g}x a false alarm.")
print(f"  cost-optimal threshold        : {thr_cost:.2f}")
print(f"  F1-optimal threshold          : {thr_f1:.2f}")
print(f"  lowest threshold at >=99% prec: {thr_p99:.2f}"
      if not np.isnan(thr_p99) else "  99% precision not attainable")
print()
display(sweep.set_index("threshold").reindex([0.10, 0.25, 0.50, 0.75, 0.90]))

OPERATING_THRESHOLD = thr_cost
evaluate(f"{CHAMPION} (default, cost threshold)", fitted[CHAMPION], X_test, y_test,
         threshold=OPERATING_THRESHOLD, stage="test / cost-optimal threshold")
print(f"Operating threshold adopted for the rest of the notebook: "
      f"{OPERATING_THRESHOLD:.2f}")
print("Note this is a *business* decision, not a statistical one: changing the")
print("FN:FP cost ratio in CFG moves the threshold and re-balances analyst")
print("workload against residual risk.")

In [ ]:
# =============================================================================
# 6.4 Where does the model fail? Recall per attack family and per label
# =============================================================================
pred_champion = (proba_champion >= OPERATING_THRESHOLD).astype("int8")
per_label = (
    meta_test.assign(detected=pred_champion, truth=y_test.to_numpy(),
                     score=proba_champion)
    .groupby(LABEL_COL, observed=True)
    .agg(rows=("detected", "size"), is_attack=("truth", "max"),
         detected=("detected", "sum"), mean_score=("score", "mean"))
)
per_label["detection_rate"] = per_label["detected"] / per_label["rows"]
per_label = per_label.sort_values(["is_attack", "detection_rate"])

print("Detection rate per label. For attack rows this is recall; for the BENIGN")
print("row it is the false-positive rate.")
display(per_label[["rows", "detected", "detection_rate", "mean_score"]])

attack_labels = per_label[per_label["is_attack"] == 1]
fig, ax = plt.subplots(figsize=(9, max(3.2, 0.42 * len(attack_labels))))
bars = ax.barh(attack_labels.index, 100 * attack_labels["detection_rate"],
               color="#e63946")
ax.set_xlim(0, 105)
ax.set_xlabel("recall (%)")
ax.set_title(f"Per-attack-family recall - {CHAMPION} @ threshold "
             f"{OPERATING_THRESHOLD:.2f}")
for bar, (rate, n) in zip(bars, zip(attack_labels["detection_rate"],
                                    attack_labels["rows"])):
    ax.text(100 * rate + 1, bar.get_y() + bar.get_height() / 2,
            f"{100 * rate:.1f}%  (n={n:,})", va="center", fontsize=8)
plt.tight_layout()
plt.show()

worst = attack_labels.head(3)
print("Weakest attack families:")
for label, row in worst.iterrows():
    print(f"  {label:<32} recall {row['detection_rate']:.1%} over "
          f"{int(row['rows']):,} test rows")
print()
print("This breakdown is the most important table in the section: an aggregate")
print("recall above 99% can still hide a family that is missed almost entirely,")
print("and the families with the fewest training examples are exactly the ones")
print("that suffer. Section 8 addresses this with anomaly detection for rare and")
print("unseen attacks, and Section 9 treats it as an equity-of-protection issue.")

---
# Section 7 - Hyperparameter tuning

## Strategy

| Decision | Choice | Justification |
| --- | --- | --- |
| Search algorithm | `RandomizedSearchCV` | With 6-7 interacting hyperparameters a full grid is combinatorially hopeless. Random search covers each marginal dimension far better than a grid for the same budget, and the budget is a single explicit number (`tuning_iters`) |
| Objective | `average_precision` (PR-AUC) | Threshold-free and minority-class focused, matching Section 6's argument. Optimising accuracy or even F1 would bias the search toward the benign class |
| Validation | `StratifiedKFold(3)` on the **training** split only | 3 folds instead of 5 buys a wider search for the same compute; stratification keeps rare attacks in every fold |
| Search data | the `cv slice` | The search fits `iters x folds` models. Searching on the slice and then refitting the winner on the full training split gives most of the benefit at a fraction of the cost |
| Leakage control | the whole `Pipeline` is the search object | Imputation and scaling are re-fitted inside every fold, so no validation statistic leaks into training |
| Both finalists tuned | not just the champion | A model's default configuration says little about its tuned ceiling; tuning both makes the final comparison fair |

## Search spaces

**HistGradientBoosting** - the classic bias/variance controls of boosting:
`learning_rate` x `max_iter` (how far and in how many steps), `max_leaf_nodes`
and `max_depth` (per-tree capacity), `min_samples_leaf` and `l2_regularization`
(smoothing), `max_bins` (histogram resolution vs speed), and `class_weight` to
counteract imbalance.

**Random Forest** - variance is already handled by bagging, so the search
targets capacity and the imbalance response: `n_estimators`, `max_depth`,
`min_samples_leaf`, `max_features`, `class_weight`, and `bootstrap`.

In [ ]:
# =============================================================================
# 7.1 Randomised search - HistGradientBoosting   (slow cell, ~4-8 min)
# =============================================================================
from scipy.stats import loguniform, randint
from sklearn.model_selection import RandomizedSearchCV

hgb_space = {
    "model__learning_rate": loguniform(0.02, 0.3),
    "model__max_iter": randint(120, 500),
    "model__max_leaf_nodes": randint(15, 96),
    "model__max_depth": [None, 6, 10, 16],
    "model__min_samples_leaf": randint(10, 120),
    "model__l2_regularization": loguniform(1e-4, 5.0),
    "model__max_bins": [128, 255],
    "model__early_stopping": [True],
    "model__validation_fraction": [0.1],
    "model__n_iter_no_change": [15],
}

# class_weight is only available on HistGradientBoostingClassifier from
# scikit-learn 1.2 onward. Probing get_params() keeps the notebook runnable on
# older runtimes instead of failing the entire search on one unknown keyword.
if "class_weight" in HistGradientBoostingClassifier().get_params():
    hgb_space["model__class_weight"] = [None, "balanced"]
    print("Searching over class_weight as well (supported by this sklearn build).")
else:
    print("This scikit-learn build has no class_weight for HistGradientBoosting;")
    print("imbalance is handled instead by the PR-AUC objective and the")
    print("cost-based threshold from Section 6.3.")

hgb_search = RandomizedSearchCV(
    estimator=make_pipeline(
        HistGradientBoostingClassifier(random_state=RANDOM_STATE), scale=False
    ),
    param_distributions=hgb_space,
    n_iter=CFG.tuning_iters,
    scoring="average_precision",
    cv=StratifiedKFold(CFG.tuning_folds, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
    verbose=1,
)

t0 = time.perf_counter()
hgb_search.fit(X_cv, y_cv)
print(f"\n{CFG.tuning_iters} configurations x {CFG.tuning_folds} folds "
      f"= {CFG.tuning_iters * CFG.tuning_folds} fits in "
      f"{time.perf_counter() - t0:.1f}s")
print(f"Best cross-validated PR-AUC: {hgb_search.best_score_:.5f}")
print("Best configuration:")
for key, value in sorted(hgb_search.best_params_.items()):
    print(f"  {key.replace('model__', ''):<22} = {value}")

hgb_results = (
    pd.DataFrame(hgb_search.cv_results_)
    .sort_values("rank_test_score")
    .set_index("rank_test_score")
)
show_cols = ["mean_test_score", "std_test_score", "mean_fit_time"] + [
    c for c in hgb_results.columns if c.startswith("param_")
]
print("\nTop 5 sampled configurations")
display(hgb_results[show_cols].head(5))
print("Worst sampled configuration (shows how much tuning actually matters)")
display(hgb_results[show_cols].tail(1))

In [ ]:
# =============================================================================
# 7.2 Randomised search - Random Forest   (slow cell, ~4-8 min)
# =============================================================================
rf_space = {
    "model__n_estimators": randint(80, 220),
    "model__max_depth": [None, 12, 20, 30],
    "model__min_samples_leaf": randint(1, 12),
    "model__min_samples_split": randint(2, 20),
    "model__max_features": ["sqrt", "log2", 0.3, 0.5],
    "model__class_weight": [None, "balanced", "balanced_subsample"],
    "model__bootstrap": [True],
}

rf_search = RandomizedSearchCV(
    estimator=make_pipeline(
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1), scale=False
    ),
    param_distributions=rf_space,
    n_iter=max(8, CFG.tuning_iters - 4),
    scoring="average_precision",
    cv=StratifiedKFold(CFG.tuning_folds, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
    verbose=1,
)

t0 = time.perf_counter()
rf_search.fit(X_cv, y_cv)
print(f"\nCompleted in {time.perf_counter() - t0:.1f}s")
print(f"Best cross-validated PR-AUC: {rf_search.best_score_:.5f}")
print("Best configuration:")
for key, value in sorted(rf_search.best_params_.items()):
    print(f"  {key.replace('model__', ''):<22} = {value}")

rf_results = (
    pd.DataFrame(rf_search.cv_results_)
    .sort_values("rank_test_score")
    .set_index("rank_test_score")
)
rf_cols = ["mean_test_score", "std_test_score", "mean_fit_time"] + [
    c for c in rf_results.columns if c.startswith("param_")
]
print("\nTop 5 sampled configurations")
display(rf_results[rf_cols].head(5))

In [ ]:
# =============================================================================
# 7.3 What did the search learn? Sensitivity to individual hyperparameters
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))

# (a) search progress: best-so-far over the sampled configurations
for label, res, colour in [("HistGradientBoosting", hgb_results, "#1d3557"),
                           ("Random Forest", rf_results, "#e63946")]:
    order = res.sort_index()["mean_test_score"].to_numpy()
    axes[0].plot(np.arange(1, len(order) + 1), np.maximum.accumulate(order),
                 marker="o", ms=3, label=label, color=colour)
axes[0].set_xlabel("configurations ranked best-first")
axes[0].set_ylabel("CV PR-AUC")
axes[0].set_title("(a) Score of the best k configurations")
axes[0].legend(fontsize=8)

# (b) sensitivity to the learning rate
axes[1].scatter(hgb_results["param_model__learning_rate"].astype(float),
                hgb_results["mean_test_score"], c="#1d3557", s=45)
axes[1].set_xscale("log")
axes[1].set_xlabel("learning_rate (log scale)")
axes[1].set_ylabel("CV PR-AUC")
axes[1].set_title("(b) HGB sensitivity to learning_rate")

# (c) accuracy vs training cost across every sampled configuration
axes[2].scatter(hgb_results["mean_fit_time"], hgb_results["mean_test_score"],
                label="HistGradientBoosting", c="#1d3557", s=45)
axes[2].scatter(rf_results["mean_fit_time"], rf_results["mean_test_score"],
                label="Random Forest", c="#e63946", s=45)
axes[2].set_xlabel("mean fit time per fold (s)")
axes[2].set_ylabel("CV PR-AUC")
axes[2].set_title("(c) Quality vs training cost")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

spread = pd.DataFrame(
    {
        "best CV PR-AUC": [hgb_results["mean_test_score"].max(),
                           rf_results["mean_test_score"].max()],
        "worst CV PR-AUC": [hgb_results["mean_test_score"].min(),
                            rf_results["mean_test_score"].min()],
        "spread": [
            hgb_results["mean_test_score"].max() - hgb_results["mean_test_score"].min(),
            rf_results["mean_test_score"].max() - rf_results["mean_test_score"].min(),
        ],
        "configs tried": [len(hgb_results), len(rf_results)],
    },
    index=["HistGradientBoosting", "Random Forest"],
)
print("How much did the configuration matter?")
display(spread)

In [ ]:
# =============================================================================
# 7.4 Refit the tuned models on the full training split and re-evaluate
# =============================================================================
tuned: dict[str, Pipeline] = {}
comparison_rows = []

for name, search in [("HistGradientBoosting", hgb_search),
                     ("Random Forest", rf_search)]:
    best = search.best_estimator_
    if name == "Random Forest":       # restore parallel fitting for the refit
        best.set_params(model__n_jobs=-1)
    t0 = time.perf_counter()
    best.fit(X_train, y_train)
    fit_s = time.perf_counter() - t0
    tuned[name] = best

    before = RESULTS[f"{name} (default)"]
    after = evaluate(f"{name} (tuned)", best, X_test, y_test,
                     stage="test / tuned hyperparameters")
    after_thr = evaluate(f"{name} (tuned, cost threshold)", best, X_test, y_test,
                         threshold=OPERATING_THRESHOLD,
                         stage="test / tuned + cost threshold")
    for tag, row in [("default", before), ("tuned", after),
                     ("tuned + cost thr", after_thr)]:
        comparison_rows.append(
            {"model": name, "configuration": tag,
             **{k: row[k] for k in ["pr_auc", "roc_auc", "f1", "recall",
                                    "precision", "mcc", "false_positive_rate",
                                    "cost_per_1k_flows"]}}
        )
    print(f"{name:<22} refit on {len(X_train):,} rows in {fit_s:5.1f}s | "
          f"PR-AUC {before['pr_auc']:.5f} -> {after['pr_auc']:.5f}")

tuning_table = pd.DataFrame(comparison_rows).set_index(["model", "configuration"])
print("\nEffect of tuning on held-out test performance")
display(tuning_table)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
plot_data = tuning_table.reset_index()
for ax, metric in zip(axes, ["pr_auc", "cost_per_1k_flows"]):
    sns.barplot(data=plot_data, x="model", y=metric, hue="configuration", ax=ax,
                palette=["#adb5bd", "#457b9d", "#1d3557"])
    ax.set_title({"pr_auc": "PR-AUC (higher is better)",
                  "cost_per_1k_flows": "Expected cost per 1,000 flows (lower is better)"}[metric])
    ax.legend(fontsize=8, title=None)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.4g", fontsize=7)
axes[0].set_ylim(plot_data["pr_auc"].min() * 0.995, 1.0)
plt.tight_layout()
plt.show()

FINAL_NAME = max(tuned, key=lambda n: RESULTS[f"{n} (tuned)"]["pr_auc"])
FINAL_MODEL = tuned[FINAL_NAME]
gain = (RESULTS[f"{FINAL_NAME} (tuned)"]["pr_auc"]
        - RESULTS[f"{FINAL_NAME} (default)"]["pr_auc"])
print(f"Final model: {FINAL_NAME} (tuned), PR-AUC gain from tuning = {gain:+.5f}")
print("The gain is small in absolute terms because the untuned ensembles are")
print("already close to the ceiling on this dataset. What tuning mainly buys")
print("here is a better cost profile and a smaller / faster model - and the")
print("search itself is the evidence that the default was not simply lucky.")

---
# Section 8 - Extra features and engineering considerations

A held-out PR-AUC is not a product. This section covers the additional work that
decides whether the model survives contact with a real network:

| # | Consideration | Question answered |
| --- | --- | --- |
| 8.1 | Feature importance | *Why* does it predict what it predicts, and are we relying on something fragile? |
| 8.2 | Feature reduction | How small can the feature set get before quality drops - i.e. how cheap can the flow meter be? |
| 8.3 | Probability calibration | Can the score be used as a risk number for queue ordering and cost decisions? |
| 8.4 | Cross-session generalisation | Does it still work on a capture session it has never seen? |
| 8.5 | Unseen / zero-day attacks | What happens for an attack family absent from training? |
| 8.6 | Adversarial robustness | How easily can an attacker evade it by reshaping traffic? |
| 8.7 | Operational readiness | Latency, throughput, model size, persistence, alert API |

In [ ]:
# =============================================================================
# 8.1 Feature importance: impurity-based vs permutation-based
# =============================================================================
from sklearn.inspection import permutation_importance

imp_frames = {}

# Impurity importance is only defined for the forest, and is known to be biased
# toward high-cardinality features - which is exactly why it is cross-checked
# against permutation importance below.
rf_model = tuned["Random Forest"].named_steps["model"]
imp_frames["impurity (RF)"] = pd.Series(
    rf_model.feature_importances_, index=FEATURES
).sort_values(ascending=False)

# Permutation importance is model agnostic and measures the metric drop when a
# single column is shuffled, so it reflects what the model actually relies on.
perm_n = min(8_000, len(X_test))
X_perm = X_test.iloc[:perm_n]
y_perm = y_test.iloc[:perm_n]
t0 = time.perf_counter()
perm = permutation_importance(
    FINAL_MODEL, X_perm, y_perm, scoring="average_precision", n_repeats=2,
    random_state=RANDOM_STATE, n_jobs=2,
)
print(f"Permutation importance on {perm_n:,} rows x {len(FEATURES)} features "
      f"in {time.perf_counter() - t0:.1f}s")
imp_frames[f"permutation ({FINAL_NAME})"] = pd.Series(
    perm.importances_mean, index=FEATURES
).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
for ax, (title, series) in zip(axes, imp_frames.items()):
    top = series.head(18).sort_values()
    ax.barh(top.index, top.values,
            color=["#e63946" if f in ENGINEERED else "#457b9d" for f in top.index])
    ax.set_title(f"Top 18 - {title}")
    ax.tick_params(labelsize=7)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in ["#e63946", "#457b9d"]]
axes[0].legend(handles, ["engineered", "raw"], loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

TOP_FEATURES = imp_frames[f"permutation ({FINAL_NAME})"].index.tolist()
agreement = len(set(TOP_FEATURES[:15]) & set(imp_frames["impurity (RF)"].index[:15]))
print(f"{agreement}/15 features appear in both top-15 lists - the two methods")
print("broadly agree, so the ranking is not an artefact of one importance measure.")
print(f"\nEngineered features in the permutation top 15: "
      f"{[f for f in TOP_FEATURES[:15] if f in ENGINEERED]}")
print("\nRisk flagged for Section 9: any reliance on port identity or absolute")
print("volume reflects this particular network's topology, not attacker")
print("behaviour, and will decay fastest when the model is deployed elsewhere.")

In [ ]:
# =============================================================================
# 8.2 How few features do we actually need?
# =============================================================================
reduction_rows = []
for k in sorted({5, 10, 20, 40, len(FEATURES)}):
    subset = TOP_FEATURES[:k]
    pipe = make_pipeline(
        HistGradientBoostingClassifier(max_iter=150, random_state=RANDOM_STATE)
    )
    t0 = time.perf_counter()
    pipe.fit(X_train[subset], y_train)
    fit_s = time.perf_counter() - t0
    t0 = time.perf_counter()
    proba = pipe.predict_proba(X_test[subset])[:, 1]
    score_s = time.perf_counter() - t0
    reduction_rows.append(
        {
            "features": k,
            "pr_auc": average_precision_score(y_test, proba),
            "recall": recall_score(y_test, (proba >= OPERATING_THRESHOLD).astype(int)),
            "fit_s": fit_s,
            "score_us_per_flow": 1e6 * score_s / len(X_test),
        }
    )
    print(f"k={k:>3} features -> PR-AUC {reduction_rows[-1]['pr_auc']:.5f} "
          f"(fit {fit_s:.1f}s)")

reduction = pd.DataFrame(reduction_rows).set_index("features")
reduction["pr_auc loss vs full"] = (
    reduction.loc[len(FEATURES), "pr_auc"] - reduction["pr_auc"]
)
display(reduction)

fig, ax1 = plt.subplots(figsize=(8, 4.2))
ax1.plot(reduction.index, reduction["pr_auc"], marker="o", color="#1d3557")
ax1.set_xlabel("number of features (permutation-ranked)")
ax1.set_ylabel("test PR-AUC", color="#1d3557")
ax2 = ax1.twinx()
ax2.plot(reduction.index, reduction["fit_s"], marker="s", ls="--", color="#e63946")
ax2.set_ylabel("training time (s)", color="#e63946")
ax2.grid(False)
plt.title("Accuracy / cost trade-off of feature reduction")
plt.tight_layout()
plt.show()

print("Engineering value: a compact feature set means a lighter flow meter on the")
print("sensor, less data collected per customer (a privacy benefit - see Section 9),")
print("faster retraining and fewer features whose collection must be justified.")

In [ ]:
# =============================================================================
# 8.3 Probability calibration - can the score be read as a risk?
# =============================================================================
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

# Calibration must be fitted on data the model did not train on, and evaluated
# on data used for neither. The test split is therefore halved.
X_cal, X_hold, y_cal, y_hold = train_test_split(
    X_test, y_test, test_size=0.5, stratify=y_test, random_state=RANDOM_STATE
)

calibrated = CalibratedClassifierCV(FINAL_MODEL, method="isotonic", cv="prefit")
calibrated.fit(X_cal, y_cal)

curves = {
    f"{FINAL_NAME} (raw)": FINAL_MODEL.predict_proba(X_hold)[:, 1],
    f"{FINAL_NAME} (isotonic)": calibrated.predict_proba(X_hold)[:, 1],
}

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="perfectly calibrated")
calib_rows = []
for label, proba in curves.items():
    frac_pos, mean_pred = calibration_curve(y_hold, proba, n_bins=10,
                                            strategy="quantile")
    axes[0].plot(mean_pred, frac_pos, marker="o", label=label)
    axes[1].hist(proba, bins=40, alpha=0.55, label=label)
    calib_rows.append(
        {
            "model": label,
            "brier": brier_score_loss(y_hold, proba),
            "pr_auc": average_precision_score(y_hold, proba),
            "mean |gap|": np.abs(frac_pos - mean_pred).mean(),
        }
    )
axes[0].set_xlabel("mean predicted probability")
axes[0].set_ylabel("observed attack frequency")
axes[0].set_title("Reliability diagram (quantile bins)")
axes[0].legend(fontsize=8)
axes[1].set_yscale("log")
axes[1].set_xlabel("predicted probability")
axes[1].set_title("Score distribution")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

display(pd.DataFrame(calib_rows).set_index("model"))
print("Why this matters operationally: the cost-optimal threshold from Section 6")
print("and any 'risk score' shown to an analyst are only meaningful if a score of")
print("0.8 really does mean roughly an 80% chance of being an attack. Tree")
print("ensembles are typically over-confident, so isotonic calibration is applied")
print("as a post-processing step - it changes the Brier score without changing the")
print("ranking (PR-AUC is essentially unchanged).")

In [ ]:
# =============================================================================
# 8.4 Cross-session generalisation: train on two captures, test on the third
# =============================================================================
sessions = sorted(df["source_file"].unique())
attack_labels_by_session = {
    s: set(df.loc[(df["source_file"] == s) & (df["is_attack"] == 1), LABEL_COL])
    for s in sessions
}
# Hold out the session contributing the most attack types that no other session
# contains: this makes the test a genuine "new network, new attacks" scenario.
holdout_session = max(
    sessions,
    key=lambda s: len(
        attack_labels_by_session[s]
        - set().union(*[attack_labels_by_session[o] for o in sessions if o != s])
    )
    if len(sessions) > 1
    else 0,
)

train_mask = df["source_file"] != holdout_session
cross_train = df[train_mask].sample(
    n=min(150_000, int(train_mask.sum())), random_state=RANDOM_STATE
)
cross_test = df[~train_mask].sample(
    n=min(80_000, int((~train_mask).sum())), random_state=RANDOM_STATE
)

train_families = set(cross_train.loc[cross_train["is_attack"] == 1, LABEL_COL])
test_families = set(cross_test.loc[cross_test["is_attack"] == 1, LABEL_COL])
unseen = sorted(test_families - train_families)

print(f"Training sessions : {[s for s in sessions if s != holdout_session]} "
      f"({len(cross_train):,} rows)")
print(f"Held-out session  : {holdout_session} ({len(cross_test):,} rows)")
print(f"Attack labels seen in training : {sorted(train_families)}")
print(f"Attack labels in the held-out session : {sorted(test_families)}")
print(f"NEVER SEEN in training : {unseen if unseen else 'none'}")

cross_model = make_pipeline(
    HistGradientBoostingClassifier(max_iter=200, random_state=RANDOM_STATE)
)
cross_model.fit(cross_train[FEATURES], cross_train["is_attack"])
cross_metrics = evaluate(
    "Cross-session (unseen capture)", cross_model, cross_test[FEATURES],
    cross_test["is_attack"], threshold=OPERATING_THRESHOLD,
    stage=f"trained without {holdout_session}",
)

random_split = pd.Series(RESULTS[f"{FINAL_NAME} (tuned, cost threshold)"])
drift = pd.DataFrame(
    {
        "random split (Section 7)": random_split,
        f"session hold-out ({holdout_session})": pd.Series(cross_metrics),
    }
).loc[["pr_auc", "roc_auc", "f1", "recall", "precision", "mcc",
       "false_positive_rate", "cost_per_1k_flows"]].astype(float)
drift["degradation"] = (
    drift["random split (Section 7)"] - drift[f"session hold-out ({holdout_session})"]
)
print()
display(drift)

if unseen:
    cross_pred = (
        cross_model.predict_proba(cross_test[FEATURES])[:, 1] >= OPERATING_THRESHOLD
    ).astype(int)
    unseen_recall = (
        cross_test.assign(detected=cross_pred)
        .query(f"{LABEL_COL} in @unseen")
        .groupby(LABEL_COL, observed=True)["detected"]
        .agg(["size", "sum", "mean"])
        .rename(columns={"size": "rows", "sum": "detected", "mean": "recall"})
    )
    print("Recall on attack types that were NEVER in the training data:")
    display(unseen_recall)

print("Interpretation: the random-split score is an optimistic upper bound. Flows")
print("from one capture session share addresses, services and timing, so a random")
print("split effectively lets the model memorise the session. The session hold-out")
print("is the honest estimate of what happens on a new customer network, and the")
print("gap is the size of the retraining / monitoring problem in production.")

In [ ]:
# =============================================================================
# 8.5 Unsupervised safety net for zero-day attacks
# =============================================================================
from sklearn.ensemble import IsolationForest

# Trained on BENIGN traffic only: it never sees an attack, so it can flag attack
# types that did not exist when the supervised model was trained.
benign_train = X_train[y_train == 0]
benign_sample = benign_train.sample(
    n=min(40_000, len(benign_train)), random_state=RANDOM_STATE
)

iso = make_pipeline(
    IsolationForest(n_estimators=150, contamination=0.02, n_jobs=-1,
                    random_state=RANDOM_STATE)
)
t0 = time.perf_counter()
iso.fit(benign_sample)
print(f"IsolationForest fitted on {len(benign_sample):,} benign-only flows in "
      f"{time.perf_counter() - t0:.1f}s")

iso_flag = (iso.predict(X_test) == -1).astype("int8")   # -1 = outlier
sup_flag = (
    FINAL_MODEL.predict_proba(X_test)[:, 1] >= OPERATING_THRESHOLD
).astype("int8")
hybrid_flag = ((iso_flag + sup_flag) > 0).astype("int8")

hybrid_rows = []
for label, flag in [("Supervised only", sup_flag),
                    ("IsolationForest only (benign-trained)", iso_flag),
                    ("Hybrid (OR rule)", hybrid_flag)]:
    tn, fp, fn, tp = confusion_matrix(y_test, flag, labels=[0, 1]).ravel()
    hybrid_rows.append(
        {
            "detector": label,
            "recall": tp / max(tp + fn, 1),
            "precision": tp / max(tp + fp, 1),
            "f1": f1_score(y_test, flag, zero_division=0),
            "false_positive_rate": fp / max(fp + tn, 1),
            "alerts_per_1k_flows": 1000 * (tp + fp) / len(y_test),
            "cost_per_1k_flows": 1000
            * (CFG.cost_false_negative * fn + CFG.cost_false_positive * fp)
            / len(y_test),
        }
    )
hybrid = pd.DataFrame(hybrid_rows).set_index("detector")
display(hybrid)

truth = y_test.to_numpy()
missed_by_supervised = (sup_flag == 0) & (truth == 1)
caught_only_by_iso = int((missed_by_supervised & (iso_flag == 1)).sum())
print(f"Attacks the anomaly detector caught that the supervised model missed: "
      f"{caught_only_by_iso:,} of {int(missed_by_supervised.sum()):,} "
      f"missed attacks")
print()
print("Design conclusion: the anomaly detector on its own is far too noisy to")
print("page an analyst, but as a second, independent channel it recovers part of")
print("the supervised model's blind spot. In production it belongs in a lower-")
print("priority 'investigate' queue rather than the blocking path, and its hits")
print("are prime candidates for labelling and inclusion in the next retraining.")

In [ ]:
# =============================================================================
# 8.6 Adversarial robustness: how easily can traffic be reshaped to evade us?
# =============================================================================
def perturb(frame: pd.DataFrame, kind: str) -> pd.DataFrame:
    '''Simulate cheap, realistic evasion tactics an attacker could apply.'''
    out = frame.copy()
    rng = np.random.default_rng(RANDOM_STATE)
    timing = [c for c in ["Flow Duration", "Flow IAT Mean", "Flow IAT Std",
                          "Flow IAT Max", "duration_s", "log_duration",
                          "packets_per_second", "bytes_per_second"]
              if c in out.columns]
    sizes = [c for c in ["Fwd Packet Length Mean", "Bwd Packet Length Mean",
                         "Average Packet Size", "Packet Length Mean",
                         "bytes_per_packet", "fwd_bytes_per_packet",
                         "Total Length of Fwd Packets"] if c in out.columns]
    if kind == "padding (+15% packet sizes)" and sizes:
        out[sizes] = out[sizes] * 1.15
    elif kind == "slow down (2x inter-arrival times)" and timing:
        out[timing] = out[timing] * 2.0
    elif kind == "jitter (10% multiplicative noise, all features)":
        noise = rng.normal(1.0, 0.10, size=out.shape).astype("float32")
        out = out * noise
    return out.astype("float32")


attack_rows = X_test[y_test == 1]
robust_rows = []
for kind in ["none (baseline)", "padding (+15% packet sizes)",
             "slow down (2x inter-arrival times)",
             "jitter (10% multiplicative noise, all features)"]:
    perturbed = attack_rows if kind == "none (baseline)" else perturb(attack_rows, kind)
    proba = FINAL_MODEL.predict_proba(perturbed)[:, 1]
    robust_rows.append(
        {
            "evasion tactic": kind,
            "recall on attacks": float((proba >= OPERATING_THRESHOLD).mean()),
            "mean attack score": float(proba.mean()),
        }
    )

robust = pd.DataFrame(robust_rows).set_index("evasion tactic")
robust["recall lost"] = robust.loc["none (baseline)", "recall on attacks"] - robust["recall on attacks"]
display(robust)

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.barh(robust.index, 100 * robust["recall on attacks"], color="#e63946")
ax.axvline(100 * robust.loc["none (baseline)", "recall on attacks"], ls="--", c="k",
           lw=1, label="baseline recall")
ax.set_xlabel("recall on attack flows (%)")
ax.set_title("Recall under simulated evasion")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("This is a lower bound on attacker effort, not a security proof: the")
print("perturbations are applied to the feature vector, so they ignore protocol")
print("constraints. Even so, any tactic that costs the model several points of")
print("recall identifies a feature the model over-trusts. Mitigations: prefer")
print("ratio features over absolute volumes (already done in Section 4), retrain")
print("on adversarially perturbed copies, and keep the independent anomaly")
print("channel from 8.5 so evasion must fool two different detectors.")

In [ ]:
# =============================================================================
# 8.7 Operational readiness: latency, size, persistence and an alert API
# =============================================================================
bench = X_test.iloc[: min(20_000, len(X_test))]

latency_rows = []
for label, model in [("Random Forest (tuned)", tuned["Random Forest"]),
                     ("HistGradientBoosting (tuned)", tuned["HistGradientBoosting"])]:
    model.predict_proba(bench.iloc[:100])          # warm-up
    t0 = time.perf_counter()
    model.predict_proba(bench)
    batch_s = time.perf_counter() - t0
    t0 = time.perf_counter()
    for i in range(200):                            # single-flow (inline) path
        model.predict_proba(bench.iloc[i : i + 1])
    single_s = (time.perf_counter() - t0) / 200
    path = os.path.join(CFG.artefact_dir, label.split(" (")[0].replace(" ", "_") + ".joblib")
    joblib.dump(model, path, compress=3)
    latency_rows.append(
        {
            "model": label,
            "batch us/flow": 1e6 * batch_s / len(bench),
            "throughput flows/s": len(bench) / batch_s,
            "single-flow ms": 1e3 * single_s,
            "size MB": os.path.getsize(path) / 1024**2,
        }
    )

latency = pd.DataFrame(latency_rows).set_index("model")
display(latency)

# --- persistence round-trip --------------------------------------------------
final_path = os.path.join(CFG.artefact_dir, "final_model.joblib")
joblib.dump({"model": FINAL_MODEL, "features": FEATURES,
             "threshold": OPERATING_THRESHOLD, "config": asdict(CFG)},
            final_path, compress=3)
bundle = joblib.load(final_path)
identical = np.allclose(
    bundle["model"].predict_proba(bench)[:, 1],
    FINAL_MODEL.predict_proba(bench)[:, 1],
)
print(f"Saved {final_path} ({os.path.getsize(final_path) / 1024**2:.1f} MB); "
      f"reloaded predictions identical: {identical}")

# --- alert API with a human-readable explanation -----------------------------
BENIGN_PROFILE = X_train[y_train == 0].median()
BENIGN_SPREAD = X_train[y_train == 0].std().replace(0, np.nan)


def score_flows(flows: pd.DataFrame, top_n: int = 3) -> pd.DataFrame:
    '''Score flows and explain each alert in terms an analyst can act on.

    The explanation lists the features that deviate most (in robust z-score
    terms) from the benign profile, restricted to the features the model
    actually relies on. This gives a per-alert justification, which is a
    requirement for both SOC workflow and the transparency obligations
    discussed in Section 9.
    '''
    proba = bundle["model"].predict_proba(flows[bundle["features"]])[:, 1]
    deviation = (
        (flows[bundle["features"]] - BENIGN_PROFILE) / BENIGN_SPREAD
    ).abs()[TOP_FEATURES[:25]]
    reasons = [
        ", ".join(
            f"{feat}={flows.iloc[i][feat]:,.3g} (z={row[feat]:.1f})"
            for feat in row.dropna().nlargest(top_n).index
        )
        for i, (_, row) in enumerate(deviation.iterrows())
    ]
    return pd.DataFrame(
        {
            "risk_score": proba.round(4),
            "verdict": np.where(proba >= bundle["threshold"], "ALERT", "allow"),
            "priority": pd.cut(proba, [-0.01, 0.5, 0.8, 0.95, 1.01],
                               labels=["none", "low", "high", "critical"]),
            "why": reasons,
        },
        index=flows.index,
    )


demo_idx = (
    list(X_test[y_test == 1].index[:3]) + list(X_test[y_test == 0].index[:2])
)
demo = score_flows(X_test.loc[demo_idx])
demo.insert(0, "true_label", meta_test.loc[demo_idx, LABEL_COL])
print("\nExample alert output (3 real attacks, 2 benign flows)")
display(demo)

## 8.8 Deployment architecture and lifecycle

```
 sensor / tap ──▶ CICFlowMeter ──▶ feature builder ──▶ scorer ──┬─▶ score ≥ 0.95 → auto-contain + notify
   (mirrored)      (flow stats)    (Section 4 code,   (joblib)  ├─▶ 0.32-0.95    → SOC triage queue
                                    shared with                  ├─▶ anomaly-only → low-priority queue
                                    training)                    └─▶ else         → log only, sampled audit
                                          │                                │
                                          └──────── feature & score logs ──┴──▶ drift monitors + labelled
                                                                                feedback → weekly retrain
```

**Non-negotiables carried over from the analysis above**

1. **One feature-building implementation** shared by training and serving.
   `engineer_features` is a pure function of a dataframe for exactly this reason;
   two implementations guarantee training/serving skew.
2. **Threshold and feature list travel with the model.** They are serialised in
   the same bundle - a model with the wrong threshold is worse than no model.
3. **Drift monitoring.** Section 8.4 quantified how much performance a *new*
   capture costs. Production therefore monitors input drift (population
   stability index per feature), score drift (alert-rate change), and outcome
   drift (analyst confirm/dismiss rate), with an alert budget rather than an
   accuracy target, since ground truth arrives late.
4. **Scheduled retraining with human-labelled feedback**, prioritising analyst
   dismissals (false positives) and anomaly-channel confirmations, which are the
   highest-information samples.
5. **Shadow deployment then staged rollout.** Every new model runs in scoring-only
   mode against live traffic and is compared with the incumbent before it can
   influence a blocking decision.
6. **Rollback.** Versioned artefacts and a pinned "last known good" model, because
   a bad detection model can take a customer's network offline.

---
# Section 9 - AI ethics considerations

A network-traffic classifier is not a neutral piece of maths. It inspects
communications, it makes accusations about people and organisations, and its
mistakes have asymmetric, sometimes invisible consequences. The nine areas below
are the ones that actually apply to *this* model, with the specific mitigation
this project adopts for each.

## 9.1 Privacy and data protection

Network metadata is personal data in most jurisdictions. Even without payload,
flow records reveal who talked to whom, when, for how long and using which
service - enough to infer an individual's employer, health enquiries, religion,
sexuality, political interest or job search.

* **Data minimisation.** Section 8.2 showed that a small feature subset retains
  almost all detection quality. That is a privacy argument as much as an
  efficiency one: collect the smallest feature set that works, and do not
  collect payload, URLs or user identifiers if statistical features suffice.
* **Pseudonymisation by design.** IP addresses and timestamps were deliberately
  *not* used as model features. They are needed for incident response, but they
  belong in an access-controlled investigation store with its own audit log, not
  in a model input vector.
* **Purpose limitation and retention.** Flow records should be retained only as
  long as the security purpose requires, with a documented schedule and hard
  deletion. Legitimate-interest processing for security does not license
  indefinite storage.
* **Lawful basis and transparency.** Employees and customers must be told that
  traffic is monitored, by whom, for what and for how long. Silent monitoring is
  the single most common failure in this product category.
* **Cross-border transfer.** If scoring happens in a different jurisdiction from
  capture, the transfer needs its own legal basis; on-premise scoring avoids the
  question entirely and is preferable for this product.

## 9.2 Fairness and disparate impact

"Fairness" here is not about protected attributes in a credit-scoring sense -
the model never sees demographics. The risk is **unequal quality of service and
unequal suspicion**:

* A false positive can block a legitimate user, so groups whose traffic looks
  unusual (research tools, VPN and Tor users, accessibility software,
  automation, older devices, night-shift workers, users on poor connections with
  odd timing profiles) absorb more of the harm. Traffic shape is a *proxy* for
  who and what someone is, so the model can end up systematically suspecting
  minority technology users - a proxy-discrimination pathway that never appears
  in an aggregate metric.
* Conversely, unequal **recall** across attack families (Section 6.4) means
  unequal *protection*: customers targeted by rare attack types get a weaker
  service than customers hit by the attacks that happen to be well represented
  in training data.

Section 9.11 audits this quantitatively, because a fairness claim without
subgroup numbers is just an assertion.

## 9.3 Transparency and explainability

An analyst who cannot see why a flow was flagged cannot overrule the model, and
a customer who is blocked deserves a reason. Blocking someone with "the model
said so" is not defensible operationally or legally. This notebook therefore
ships global importances (8.1), a per-alert explanation in the alert API (8.7),
and a model card (below). Tree ensembles were also preferred partly *because*
they support this; a marginal accuracy gain from an opaque model would be a poor
trade here.

## 9.4 Accountability and human oversight

* **Human in the loop for consequential actions.** Automated blocking is limited
  to the highest-confidence band, is reversible, and is logged; everything else
  becomes a recommendation to a human.
* **Named ownership.** A specific team owns the model's behaviour - "the
  algorithm decided" is not an acceptable answer to an affected customer.
* **Contestability.** There must be a route for a customer or employee to
  challenge a block and have it reviewed by a person, with the outcome fed back
  into training data.
* **Audit trail.** Model version, feature values, score, threshold and the acting
  identity are recorded for every consequential decision.

## 9.5 Security of the model itself

The model is part of the attack surface it defends:

* **Evasion** (Section 8.6) - attackers reshape traffic to look benign.
* **Poisoning** - if analyst feedback is used for retraining, an attacker who can
  generate traffic and influence labels can teach the model to ignore them.
  Mitigation: no unreviewed automated labels, anomaly screening of new training
  data, and a canary evaluation set that must not regress.
* **Model extraction / probing** - repeated probing reveals the decision boundary.
  Mitigation: rate limiting, no score exposure to untrusted parties.
* **Dependence risk** - over-trust in an ML detector degrades other controls.
  It must augment, not replace, defence in depth.

## 9.6 Dataset bias, construct validity and honest limitations

This is the most important ethical point in the whole notebook, because it
governs what may truthfully be *claimed*:

* The data comes from a **controlled capture** with scripted attacks. Real
  networks are noisier and real attackers are less uniform, so the reported
  scores are an **upper bound**, not a production expectation. Section 8.4
  measures part of this gap directly.
* **Label quality.** Labels were assigned by capture window, so mislabelling at
  the boundaries is likely; "ground truth" is itself an estimate.
* **Topology leakage.** Port and volume features encode where the attacks were
  aimed in the lab. Advertising the resulting score as generalising to a
  customer network would be misleading.
* **Temporal validity.** Attack techniques and normal traffic both change.
  A model trained on an older capture decays, and undetected decay is a silent
  safety failure. Hence monitoring and scheduled retraining, not a one-off
  evaluation.
* **Publication honesty.** Marketing should quote the session hold-out figure
  with its caveats, not the flattering random-split figure.

## 9.7 Dual use, scope creep and proportionality

The same model that flags a botnet can flag "an employee used a VPN",
"this device is a whistle-blower's" or "this person is job hunting". Traffic
classifiers are readily repurposed into employee surveillance or censorship
tooling.

* Contractual and technical **scope limits**: security use only, with logged
  access.
* **Proportionality review** before adding any feature that increases
  intrusiveness (e.g. deep packet inspection, TLS interception, per-user
  profiling).
* **Refusal criteria**: XYZ should be willing to decline deployments whose
  evident purpose is monitoring individuals rather than defending infrastructure.

## 9.8 Regulatory and standards context

* **GDPR / UK GDPR / equivalent regimes** (e.g. Malaysia's PDPA, Singapore's
  PDPA): lawful basis, minimisation, retention limits, transparency, DPIA for
  systematic monitoring, and Article 22-style safeguards for automated decisions
  with significant effects.
* **EU AI Act**: general-purpose cybersecurity analytics is not high-risk *per
  se*, but the same model applied to employee monitoring would fall into the
  employment high-risk category, and the deployment context - not the code -
  decides. Documentation, logging, human oversight and risk management should be
  built to that standard now rather than retrofitted.
* **Standards to align with**: ISO/IEC 27001 for the surrounding controls, the
  NIST AI Risk Management Framework for lifecycle governance, and model/data
  cards for documentation.

## 9.9 Environmental and economic cost

The randomised searches in Section 7 are, by design, the most compute-intensive
part of this notebook. Budgeted search (random over grid, 3 folds, a stratified
subsample, early stopping) was chosen partly to keep energy use and cost
proportionate to the gain - which Section 7.4 showed to be small. Retraining
cadence should be justified by measured drift, not run nightly out of habit.

## 9.10 Governance checklist adopted for this project

| Control | Status in this notebook |
| --- | --- |
| Subgroup performance audit before release | implemented in 9.11 |
| Per-alert explanation available to analysts | implemented in 8.7 |
| Model card published with the artefact | implemented in 9.12 |
| Human review for consequential actions | designed in 8.8 |
| Documented limitations, incl. optimistic-benchmark warning | Sections 8.4, 9.6, 10 |
| Data minimisation evidence | Section 8.2 |
| Drift monitoring and rollback plan | Section 8.8 |
| DPIA, retention schedule, access control | **required before deployment - outside the scope of this notebook** |

In [ ]:
# =============================================================================
# 9.11 Quantitative fairness / disparate-impact audit
# =============================================================================
# Equal aggregate performance can hide very unequal performance per subgroup.
# Groups here are operational rather than demographic - the model never sees
# demographics - but they are the channels through which unequal treatment of
# real users and customers would actually appear.
audit = meta_test.copy()
audit["truth"] = y_test.to_numpy()
audit["flag"] = (FINAL_MODEL.predict_proba(X_test)[:, 1] >= OPERATING_THRESHOLD).astype(int)

# (a) service group - which kinds of legitimate users bear the false positives
if "Destination Port" in audit.columns:
    port_lookup = {p: name for name, ports in WELL_KNOWN_PORTS.items() for p in ports}
    audit["service group"] = (
        audit["Destination Port"].astype("int64").map(port_lookup).fillna("other/high port")
    )
else:
    audit["service group"] = "unknown"

# (b) flow size - small, low-volume flows are the hardest to judge
audit["flow size"] = pd.qcut(
    work.loc[X_test.index, "total_bytes"], q=4,
    labels=["tiny", "small", "medium", "large"], duplicates="drop",
)

# (c) capture session - a stand-in for "different customer network"
audit["session"] = audit["source_file"]


def subgroup_audit(frame: pd.DataFrame, by: str, min_rows: int = 200) -> pd.DataFrame:
    '''Per-subgroup error rates: who pays for this model's mistakes?'''
    rows = []
    for group, part in frame.groupby(by, observed=True):
        if len(part) < min_rows:
            continue
        tn, fp, fn, tp = confusion_matrix(part["truth"], part["flag"],
                                          labels=[0, 1]).ravel()
        rows.append(
            {
                by: str(group),
                "rows": len(part),
                "attack %": 100 * part["truth"].mean(),
                "FPR % (legit traffic blocked)": 100 * fp / max(fp + tn, 1),
                "FNR % (attacks missed)": 100 * fn / max(fn + tp, 1),
                "precision": tp / max(tp + fp, 1),
            }
        )
    return pd.DataFrame(rows).set_index(by).sort_values(
        "FPR % (legit traffic blocked)", ascending=False
    )


fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for ax, dimension in zip(axes, ["service group", "flow size", "session"]):
    table = subgroup_audit(audit, dimension)
    print(f"\nSubgroup audit by {dimension}")
    display(table)
    if table.empty:
        continue
    idx = np.arange(len(table))
    ax.barh(idx - 0.2, table["FPR % (legit traffic blocked)"], height=0.4,
            color="#e63946", label="FPR % (legit blocked)")
    ax.barh(idx + 0.2, table["FNR % (attacks missed)"], height=0.4,
            color="#1d3557", label="FNR % (attacks missed)")
    ax.set_yticks(idx, table.index, fontsize=8)
    ax.set_title(f"Error rates by {dimension}")
    ax.set_xlabel("%")
    ax.legend(fontsize=7)
    nonzero = table["FPR % (legit traffic blocked)"].replace(0, np.nan).dropna()
    if len(nonzero) > 1:
        print(f"  worst/best false-positive-rate ratio: "
              f"{nonzero.max() / nonzero.min():.1f}x")
plt.tight_layout()
plt.show()

print("How to read this audit")
print("- A high FPR in one service group means legitimate users of that service")
print("  absorb a disproportionate share of blocks: that is the disparate-impact")
print("  channel for this kind of model, and a per-group threshold or a")
print("  group-aware review rule is the mitigation.")
print("- A high FNR in one group means customers of that group are under-")
print("  protected: unequal safety rather than unequal burden.")
print("- Release criterion adopted: no subgroup with >=200 test flows may have a")
print("  false-positive rate more than 3x the best subgroup without a documented,")
print("  signed-off justification.")

In [ ]:
# =============================================================================
# 9.12 Model card - the documentation that ships with the artefact
# =============================================================================
final_metrics = RESULTS[f"{FINAL_NAME} (tuned, cost threshold)"]
cross_ref = RESULTS["Cross-session (unseen capture)"]

MODEL_CARD = {
    "Model name": f"xyz-flow-threat-detector / {FINAL_NAME}",
    "Version": "0.1 (research prototype - NOT production approved)",
    "Owner": "XYZ Cybersecurity, Detection Engineering team",
    "Date": pd.Timestamp.today().strftime("%Y-%m-%d"),
    "Intended use": "Triage support: rank network flows by likelihood of being "
                    "malicious, to prioritise human SOC investigation.",
    "Out-of-scope use": "Autonomous blocking outside the highest-confidence band; "
                        "employee monitoring; individual profiling; evidence in "
                        "disciplinary or legal proceedings; deployment on a network "
                        "unlike the training network without revalidation.",
    "Inputs": f"{len(FEATURES)} numeric flow-statistics features "
              f"({len([f for f in FEATURES if f in ENGINEERED])} engineered). "
              "No IP addresses, no payload, no user identifiers.",
    "Output": "Calibrated probability of maliciousness + decision at threshold "
              f"{OPERATING_THRESHOLD:.2f} (set from an explicit "
              f"{CFG.cost_false_negative:g}:{CFG.cost_false_positive:g} FN:FP cost ratio).",
    "Training data": f"{len(X_train):,} flows, stratified sample of "
                     f"{len(df):,} cleaned flows from {df['source_file'].nunique()} "
                     "capture sessions of a controlled lab network.",
    "Random-split performance": f"PR-AUC {final_metrics['pr_auc']:.4f}, "
                                f"recall {final_metrics['recall']:.4f}, "
                                f"precision {final_metrics['precision']:.4f}, "
                                f"MCC {final_metrics['mcc']:.4f}",
    "Unseen-session performance (the honest number)":
        f"PR-AUC {cross_ref['pr_auc']:.4f}, recall {cross_ref['recall']:.4f}, "
        f"precision {cross_ref['precision']:.4f}",
    "Known weakest classes": ", ".join(
        f"{lab} ({rate:.0%})" for lab, rate in
        per_label[per_label["is_attack"] == 1]["detection_rate"].head(3).items()
    ),
    "Ethical risks": "Privacy (traffic metadata is personal data); disparate "
                     "false-positive burden across service groups (audited in 9.11); "
                     "unequal protection across attack families; dual-use "
                     "surveillance potential; adversarial evasion.",
    "Mitigations": "Data minimisation; no identifiers as features; per-alert "
                   "explanations; human review for consequential actions; subgroup "
                   "release criterion; drift monitoring; independent anomaly channel; "
                   "documented limitations.",
    "Fairness audit": "Per-service-group, per-flow-size and per-session error "
                      "rates reported in Section 9.11.",
    "Maintenance": "Retrain on measured drift; shadow-deploy every candidate; "
                   "versioned artefacts with pinned rollback.",
    "Contact / redress": "security-ml-owner@xyz.example - customers and employees "
                         "may contest a block and obtain human review.",
}

print("=" * 78)
print("MODEL CARD".center(78))
print("=" * 78)
for key, value in MODEL_CARD.items():
    print(f"\n{key}")
    text = str(value)
    while len(text) > 74:
        cut = text.rfind(" ", 0, 74)
        cut = cut if cut > 0 else 74
        print("   " + text[:cut])
        text = text[cut:].lstrip()
    print("   " + text)
print("\n" + "=" * 78)

card_path = os.path.join(CFG.artefact_dir, "model_card.json")
pd.Series(MODEL_CARD).to_json(card_path, indent=2)
print(f"Model card written to {card_path}")

---
# Section 10 - Consolidated results and conclusions

## 10.1 Every experiment in one table

`RESULTS` has been populated by each stage as it ran, so the table below is
assembled from the actual scored runs rather than transcribed by hand.

In [ ]:
# =============================================================================
# 10.1 Consolidated results from every experiment in this notebook
# =============================================================================
summary = pd.DataFrame(RESULTS).T
metric_order = [c for c in ["stage", "threshold", "pr_auc", "roc_auc", "f1",
                            "recall", "precision", "mcc", "balanced_accuracy",
                            "false_positive_rate", "cost_per_1k_flows"]
                if c in summary.columns]
summary = summary[metric_order]
numeric_cols = [c for c in summary.columns if c != "stage"]
summary[numeric_cols] = summary[numeric_cols].apply(pd.to_numeric, errors="coerce")

print("All experiments, in the order they were run")
display(summary.sort_values("pr_auc", ascending=False))

results_path = os.path.join(CFG.artefact_dir, "all_results.csv")
summary.to_csv(results_path)
print(f"Written to {results_path}")

headline = pd.DataFrame(
    {
        "metric": ["PR-AUC", "recall", "precision", "false-positive rate",
                   "MCC", "cost per 1k flows"],
        "random split (optimistic)": [
            final_metrics["pr_auc"], final_metrics["recall"],
            final_metrics["precision"], final_metrics["false_positive_rate"],
            final_metrics["mcc"], final_metrics["cost_per_1k_flows"],
        ],
        "unseen session (realistic)": [
            cross_ref["pr_auc"], cross_ref["recall"], cross_ref["precision"],
            cross_ref["false_positive_rate"], cross_ref["mcc"],
            cross_ref["cost_per_1k_flows"],
        ],
    }
).set_index("metric")
print(f"\nHeadline result - {FINAL_NAME} (tuned) at threshold "
      f"{OPERATING_THRESHOLD:.2f}")
display(headline)

## 10.2 What was built

A complete, reproducible detection pipeline: schema harmonisation and quality
repair of three capture sessions → leakage-free cleaning → ~40 engineered
scale-invariant flow features with correlation pruning → an eight-candidate
cross-validated model comparison → imbalance-aware evaluation with a
cost-derived operating threshold → randomised hyperparameter search on the two
finalists → and the surrounding engineering (importance analysis, feature
reduction, calibration, session-level generalisation test, zero-day anomaly
channel, evasion testing, latency benchmarking, serialisation, alert API, model
card and fairness audit).

## 10.3 What the numbers actually support

* Tree ensembles clearly beat the linear and naive-Bayes baselines, which is
  direct evidence that the decision boundary in flow-feature space is strongly
  non-linear. The `DummyClassifier` row is the proof that accuracy must be
  discarded as a decision metric on this data.
* Engineered ratio/rate features occupy much of the top of both the mutual
  information and the permutation importance rankings, so the feature
  engineering contributed real signal rather than volume.
* Hyperparameter tuning produced a **small** gain, because the untuned ensembles
  were already near the ceiling on this dataset. Reporting that honestly is more
  useful than presenting the search as transformative; its real value was
  confirming the default was not luck, and improving the cost profile.
* The **random-split score is not the deployable estimate**. The session
  hold-out in 8.4 is the number the client should plan around, and the
  difference is the cost of topology and session memorisation.

## 10.4 Limitations

1. **Benchmark optimism.** Scripted attacks in a controlled capture; real
   traffic is noisier and real adversaries adapt.
2. **Label provenance.** Window-based labelling means boundary flows are
   probably mislabelled, which caps measurable precision.
3. **Sampling.** A stratified sample of ~250k rows is used so the notebook runs
   in Colab. Class proportions are preserved, so estimates are unbiased, but
   confidence intervals are wider than with the full ~10^6 rows. Raise
   `CFG.model_sample` to trade runtime for precision.
4. **Binary framing.** Operationally correct, but it does not tell the analyst
   *which* attack it is; the per-family breakdown is a partial answer only.
5. **Rare-family performance is weak** exactly where data is thin - the families
   with the fewest examples are the least reliably detected.
6. **No temporal validation.** A production system should be validated with a
   time-based split (train on the past, test on the future); the available
   captures are too short to do this properly.
7. **Evasion testing is feature-space only** and therefore an optimistic view of
   attacker effort.

## 10.5 Future work, in priority order

1. **Time-based and multi-network validation** - the only way to get a
   trustworthy production estimate.
2. **Hierarchical detection** - binary gate for speed, then a multi-class stage
   on flagged flows to name the attack family and route the response.
3. **Cost-sensitive learning and per-group thresholds** instead of a single
   global threshold, driven by the audit in 9.1.
4. **Sequence and graph context** - a flow in isolation loses the scan/lateral
   movement pattern that host- or session-level aggregation would reveal.
5. **Semi-supervised learning from the anomaly channel** to keep up with novel
   attacks between labelled retrains.
6. **Adversarial training** against protocol-valid perturbations.
7. **Governance completion** - DPIA, retention schedule, access controls and the
   shadow-deployment gate before any customer traffic is scored.

---

## References

- Sharafaldin, I., Habibi Lashkari, A. & Ghorbani, A. (2018). *Toward Generating
  a New Intrusion Detection Dataset and Intrusion Traffic Characterization.*
  ICISSP. (CICIDS2017 dataset and CICFlowMeter feature definitions.)
- Pedregosa, F. et al. (2011). *Scikit-learn: Machine Learning in Python.* JMLR.
- Bergstra, J. & Bengio, Y. (2012). *Random Search for Hyper-Parameter
  Optimization.* JMLR.
- Saito, T. & Rehmsmeier, M. (2015). *The Precision-Recall Plot Is More
  Informative than the ROC Plot When Evaluating Binary Classifiers on Imbalanced
  Datasets.* PLOS ONE.
- Chicco, D. & Jurman, G. (2020). *The advantages of the Matthews correlation
  coefficient (MCC) over F1 score and accuracy.* BMC Genomics.
- Mitchell, M. et al. (2019). *Model Cards for Model Reporting.* ACM FAT*.
- Arp, D. et al. (2022). *Dos and Don'ts of Machine Learning in Computer
  Security.* USENIX Security.
- NIST (2023). *Artificial Intelligence Risk Management Framework (AI RMF 1.0).*